# EXP24 — Proposal-Aligned Neighbor Sampling for GraphSAGE
## Jaccard + ASA-GNN Attribute Homophily & Degree + Local PPR
Mengeksekusi paired multi-seed (5 seed: 42, 100, 2026, 3407, 8713) dengan telemetri per-epoch granular.


In [ ]:
%%writefile requirements.txt
torch>=2.0.0
torch_geometric>=2.3.0
# Catatan untuk weighted neighbor sampling ('topology' dan 'importance'):
# pyg_lib diinstal dengan URL wheel sesuai versi CUDA/PyTorch, contoh:
# pip install pyg_lib -f https://data.pyg.org/whl/torch-2.6.0+cu126.html
xgboost>=1.7.0
scikit-learn>=1.2.0
pandas>=2.0.0
numpy>=1.24.0
psutil>=5.9.0
pyyaml>=6.0
matplotlib>=3.7.0


In [ ]:
%%writefile config.reference.yaml
experiment:
  id: "EXP24-ProposalSampler"
  run_id: null # Optional custom Run ID (null = auto: exp24_{sampling}_s{seed}_{timestamp})
  version: "v1_proposal_aligned"
  seed: 42
  evaluation_seed: 1042
  seeds: [42, 100, 2026, 3407, 8713]

data:
  transactions: "../../dataset/credit_card_transactions-ibm_v2.csv"
  max_rows: null
  start_year: 2016

time_window:
  reference_date: "2018-01-01"
  train_days: 300
  val_days: 30
  gap_days: 8
  test_days: 60

features:
  mode: "clean"
  use_cyclical_time: true
  use_amount_log: true
  exclude_high_cardinality: true

sampling:
  method: "uniform"
  undersampling_ratio: 1.0
  weight_attr: "sampling_weight"
  smoothing_alpha: 0.1
  topology_alpha: 0.5
  importance_gamma: 0.3
  ppr_teleport: 0.15
  ppr_steps: 3

model:
  hidden_channels: 128
  out_channels: 1
  num_layers: 2
  aggr: "mean"
  dropout: 0.2

training:
  batch_size: 1024
  epochs: 20
  learning_rate: 0.001
  weight_decay: 0.0001
  label_smoothing: 0.0
  grad_clip_norm: 5.0
  warmup_epochs: 0
  use_lr_scheduler: true
  lr_scheduler_factor: 0.5
  lr_scheduler_patience: 3
  early_stopping_patience: 7
  loss_weight_strategy: "sqrt"
  max_pos_weight: 50.0
  log_step_interval: 1
  num_neighbors: [15, 10]

classifier:
  type: "xgboost"
  n_estimators: 100
  max_depth: 4
  learning_rate: 0.05
  min_child_weight: 10
  subsample: 0.8
  colsample_bytree: 0.8
  scale_pos_weight_strategy: "sqrt"
  tree_method: "hist"

threshold_calibration:
  selection_metric: "val_f1"
  min_precision_constraint: 0.15

operational:
  budgets: [100, 200, 300, 421, 500, 1000]

reproducibility:
  deterministic: true
  warn_only: true

execution:
  seeds_to_run: [42, 100, 2026, 3407, 8713]
  resume_completed: true
  run_tabular_baseline_once: true

logging:
  log_to_file: true
  log_dir: "../../result/exp24_proposal_sampler/logs"
  log_level: "INFO"

output:
  model_dir: "../../model/exp24_proposal_sampler"
  result_dir: "../../result/exp24_proposal_sampler"
  runs_dir: "../../result/exp24_proposal_sampler/runs"
  isolate_runs: true
  save_feature_matrices: false


In [ ]:
%%writefile config.kaggle.yaml
experiment:
  id: "EXP24-ProposalSampler"
  run_id: null # Optional custom Run ID (null = auto: exp24_{sampling}_s{seed}_{timestamp})
  version: "v1_proposal_aligned"
  seed: 42
  evaluation_seed: 1042
  seeds: [42, 100, 2026, 3407, 8713]

data:
  transactions: "/kaggle/input/credit-card-transactions-ibm/credit_card_transactions-ibm_v2.csv"
  max_rows: null
  start_year: 2016

time_window:
  reference_date: "2018-01-01"
  train_days: 300
  val_days: 30
  gap_days: 8
  test_days: 60

features:
  mode: "clean"
  use_cyclical_time: true
  use_amount_log: true
  exclude_high_cardinality: true

sampling:
  method: "uniform"
  undersampling_ratio: 1.0
  weight_attr: "sampling_weight"
  smoothing_alpha: 0.1
  topology_alpha: 0.5
  importance_gamma: 0.3
  ppr_teleport: 0.15
  ppr_steps: 3

model:
  hidden_channels: 128
  out_channels: 1
  num_layers: 2
  aggr: "mean"
  dropout: 0.2

training:
  batch_size: 1024
  epochs: 20
  learning_rate: 0.001
  weight_decay: 0.0001
  label_smoothing: 0.0
  grad_clip_norm: 5.0
  warmup_epochs: 0
  use_lr_scheduler: true
  lr_scheduler_factor: 0.5
  lr_scheduler_patience: 3
  early_stopping_patience: 7
  loss_weight_strategy: "sqrt"
  max_pos_weight: 50.0
  log_step_interval: 1
  num_neighbors: [15, 10]

classifier:
  type: "xgboost"
  n_estimators: 100
  max_depth: 4
  learning_rate: 0.05
  min_child_weight: 10
  subsample: 0.8
  colsample_bytree: 0.8
  scale_pos_weight_strategy: "sqrt"
  tree_method: "hist"

threshold_calibration:
  selection_metric: "val_f1"
  min_precision_constraint: 0.15

operational:
  budgets: [100, 200, 300, 421, 500, 1000]

reproducibility:
  deterministic: true
  warn_only: true

execution:
  seeds_to_run: [42, 100, 2026, 3407, 8713]
  resume_completed: true
  run_tabular_baseline_once: true

logging:
  log_to_file: true
  log_dir: "/kaggle/working/result/exp24_proposal_sampler/logs"
  log_level: "INFO"

output:
  model_dir: "/kaggle/working/model/exp24_proposal_sampler"
  result_dir: "/kaggle/working/result/exp24_proposal_sampler"
  runs_dir: "/kaggle/working/result/exp24_proposal_sampler/runs"
  isolate_runs: true
  save_feature_matrices: false


In [ ]:
%%writefile config.smoke.yaml
# ==============================================================================
# EXP24: PROPOSAL-ALIGNED NEIGHBOR SAMPLING FOR GRAPHSAGE — SMOKE TEST CONFIG
# ==============================================================================
# Konfigurasi ini dirancang khusus untuk verifikasi cepat (smoke test) end-to-end:
# 1. Menggunakan jendela waktu proposal penuh: 300 hari Train, 30 hari Val, 8 hari Gap, 60 hari Test.
# 2. Membatasi data pada max_rows: 180000 (era modern >= 2016).
#    Pada 180.000 baris ini:
#    - Train (300 hari): ~35.840 transaksi, 67 fraud kasus
#    - Val   (30 hari) : ~3.585 transaksi, 5 fraud kasus (AUPRC valid & non-zero!)
#    - Test  (60 hari) : ~7.060 transaksi, 18 fraud kasus (F1, Precision, Recall non-zero!)
# 3. 10 Epoch pelatihan GNN (~8-10 detik di GPU RTX 5060 Ti).
# 4. 50 Estimator XGBoost (~1 detik).
# ==============================================================================

experiment:
  id: "EXP24-ProposalSampler-Smoke"
  run_id: null # String ID unik (null = auto: exp24_{sampling}_s{seed}_{timestamp})
  version: "v1_smoke_test"
  seed: 42
  evaluation_seed: 1042
  seeds: [42, 100, 2026, 3407, 8713]

data:
  transactions: "../../dataset/credit_card_transactions-ibm_v2.csv"
  max_rows: 180000 # [SMOKE] 180k baris terfilter >= 2016 (cepat & memuat fraud di semua split)
  start_year: 2016

time_window:
  reference_date: "2018-01-01"
  train_days: 300 # Sesuai rencana proposal
  val_days: 30   # Sesuai rencana proposal
  gap_days: 8    # Sesuai rencana proposal
  test_days: 60  # Sesuai rencana proposal

features:
  mode: "clean"
  use_cyclical_time: true
  use_amount_log: true
  exclude_high_cardinality: true

sampling:
  method: "topology" # Default sampler untuk smoke test
  undersampling_ratio: 1.0
  weight_attr: "sampling_weight"
  smoothing_alpha: 0.1
  topology_alpha: 0.5
  importance_gamma: 0.3
  ppr_teleport: 0.15
  ppr_steps: 3

model:
  hidden_channels: 128
  out_channels: 1
  num_layers: 2
  aggr: "mean"
  dropout: 0.2

training:
  batch_size: 1024
  epochs: 10 # [SMOKE] 10 epoch untuk pengujian kilat
  learning_rate: 0.001
  weight_decay: 0.0001
  label_smoothing: 0.0
  grad_clip_norm: 5.0
  warmup_epochs: 0
  use_lr_scheduler: true
  lr_scheduler_factor: 0.5
  lr_scheduler_patience: 3
  early_stopping_patience: 5
  loss_weight_strategy: "sqrt"
  max_pos_weight: 50.0
  log_step_interval: 1
  num_neighbors: [15, 10]

classifier:
  type: "xgboost"
  n_estimators: 50 # [SMOKE] 50 pohon agar training XGBoost ekstra cepat
  max_depth: 4
  learning_rate: 0.05
  min_child_weight: 10
  subsample: 0.8
  colsample_bytree: 0.8
  scale_pos_weight_strategy: "sqrt"
  tree_method: "hist"

threshold_calibration:
  selection_metric: "val_f1"
  min_precision_constraint: 0.15

operational:
  budgets: [100, 200, 300, 421, 500, 1000]

reproducibility:
  deterministic: true
  warn_only: true

execution:
  seeds_to_run: [42] # [SMOKE] Default seed 42 untuk uji kilat (bisa dipanggil --seeds 42 100...)
  resume_completed: false
  run_tabular_baseline_once: true

logging:
  log_to_file: true
  log_dir: "../../result/exp24_proposal_sampler/logs"
  log_level: "INFO"

output:
  model_dir: "../../model/exp24_proposal_sampler"
  result_dir: "../../result/exp24_proposal_sampler"
  runs_dir: "../../result/exp24_proposal_sampler/runs"
  isolate_runs: true
  save_feature_matrices: false


In [ ]:
%%writefile config.py
from pathlib import Path
import yaml


def load_config(config_path: str = "config.reference.yaml") -> dict:
    path = Path(config_path)
    if not path.exists():
        raise FileNotFoundError(f"Berkas konfigurasi tidak ditemukan: {config_path}")
    with path.open("r", encoding="utf-8") as handle:
        config = yaml.safe_load(handle)
    if config.get("training", {}).get("num_neighbors") != [15, 10]:
        raise ValueError("EXP24 mengunci num_neighbors=[15, 10]")
    if config.get("sampling", {}).get("weight_attr") != "sampling_weight":
        raise ValueError("EXP24 mewajibkan weight_attr='sampling_weight'")
    return config


In [ ]:
%%writefile data_preprocessing.py
import pandas as pd
import numpy as np
from datetime import timedelta
from sklearn.preprocessing import StandardScaler

def load_and_split_raw_data(
    file_path: str, 
    start_year: int = 2016, 
    nrows: int = None,
    reference_date: str = '2018-01-01',
    train_days: int = 320,
    val_days: int = 40,
    gap_days: int = 8,
    test_days: int = 40
) -> tuple:
    """
    Memuat dataset mentah IBM Credit Card Transactions, memfilter era modern (Year >= start_year),
    melakukan parsing datetime kronologis, dan memotongnya ke dalam 4 partisi temporal SHINE FRAUD II
    SEBELUM proses rekayasa fitur atau penskalaan (Strict Out-of-Sample Protocol).
    """
    print(f"Loading data from {file_path} (Filter: Year >= {start_year}, nrows={nrows})...")
    
    # 1. Pemuatan chunked efisien jika start_year ditentukan
    if start_year is not None:
        chunks = []
        collected_rows = 0
        chunksize = 1_000_000
        
        for chunk in pd.read_csv(file_path, chunksize=chunksize):
            filtered_chunk = chunk[chunk['Year'] >= start_year]
            if len(filtered_chunk) > 0:
                if nrows is not None:
                    remaining = nrows - collected_rows
                    if remaining <= 0:
                        break
                    filtered_chunk = filtered_chunk.iloc[:remaining]
                chunks.append(filtered_chunk)
                collected_rows += len(filtered_chunk)
                if nrows is not None and collected_rows >= nrows:
                    break
                    
        if not chunks:
            raise ValueError(f"Tidak ada baris yang ditemukan dengan Year >= {start_year} pada file {file_path}")
            
        df = pd.concat(chunks, ignore_index=True)
    else:
        df = pd.read_csv(file_path, nrows=nrows)

    print(f"Memproses {len(df)} baris data mentah...")
    print("Parsing timestamps...")
    df['Time'] = df['Time'].fillna('00:00')
    df['datetime'] = pd.to_datetime(
        df['Year'].astype(str) + '-' + 
        df['Month'].astype(str).str.zfill(2) + '-' + 
        df['Day'].astype(str).str.zfill(2) + ' ' + 
        df['Time']
    )
    
    print("Sorting chronologically...")
    df = df.sort_values('datetime').reset_index(drop=True)
    
    print("Cleaning Amount & Target Label...")
    df['Amount'] = df['Amount'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)
    df['Is Fraud?'] = df['Is Fraud?'].map({'Yes': 1, 'No': 0})
    
    # 2. Pembagian Jendela Waktu Temporal (SHINE FRAUD II)
    print("\nMelakukan pembagian temporal window strictly chronologic...")
    t0 = pd.to_datetime(reference_date)
    t_val_start = t0 + timedelta(days=train_days)
    t_gap_start = t_val_start + timedelta(days=val_days)
    t_test_start = t_gap_start + timedelta(days=gap_days)
    t_test_end = t_test_start + timedelta(days=test_days)
    
    train_df = df[(df['datetime'] >= t0) & (df['datetime'] < t_val_start)].copy().reset_index(drop=True)
    val_df = df[(df['datetime'] >= t_val_start) & (df['datetime'] < t_gap_start)].copy().reset_index(drop=True)
    gap_df = df[(df['datetime'] >= t_gap_start) & (df['datetime'] < t_test_start)].copy().reset_index(drop=True)
    test_df = df[(df['datetime'] >= t_test_start) & (df['datetime'] < t_test_end)].copy().reset_index(drop=True)
    
    def compute_split_stats(name: str, split_df: pd.DataFrame, start_dt, end_dt):
        n_tx = len(split_df)
        if n_tx == 0:
            return {
                "name": name,
                "start_date": str(start_dt),
                "end_date": str(end_dt),
                "num_transactions": 0,
                "num_fraud": 0,
                "fraud_rate_pct": 0.0,
                "unique_users": 0,
                "unique_merchants": 0,
                "total_nodes": 0
            }
        n_fraud = int(split_df['Is Fraud?'].sum())
        n_users = int(split_df['User'].nunique())
        n_merchants = int(split_df['Merchant Name'].nunique())
        return {
            "name": name,
            "start_date": str(start_dt),
            "end_date": str(end_dt),
            "num_transactions": n_tx,
            "num_fraud": n_fraud,
            "fraud_rate_pct": round((n_fraud / n_tx) * 100, 4),
            "unique_users": n_users,
            "unique_merchants": n_merchants,
            "total_nodes": n_tx + n_users + n_merchants
        }

    total_window_df = pd.concat([train_df, val_df, gap_df, test_df], ignore_index=True)
    
    stats_dict = {
        "protocol": "SHINE FRAUD II Rolling Window Alignment (EXP21D Golden Master)",
        "reference_date_T0": str(t0),
        "durations_days": {
            "train": train_days,
            "validation": val_days,
            "embargo_gap": gap_days,
            "test": test_days,
            "total_horizon": train_days + val_days + gap_days + test_days
        },
        "splits": {
            "train": compute_split_stats("Train", train_df, t0, t_val_start),
            "validation": compute_split_stats("Validation", val_df, t_val_start, t_gap_start),
            "embargo_gap": compute_split_stats("Embargo Gap", gap_df, t_gap_start, t_test_start),
            "test": compute_split_stats("Test", test_df, t_test_start, t_test_end),
            "total_window": compute_split_stats("Total Window", total_window_df, t0, t_test_end)
        }
    }
    
    return train_df, val_df, test_df, gap_df, stats_dict

def apply_strict_anti_leakage_preprocessing(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    gap_df: pd.DataFrame = None,
    feature_mode: str = "clean",
    use_cyclical_time: bool = True,
    use_amount_log: bool = True
) -> tuple:
    """
    Menerapkan Strict Anti-Leakage Preprocessing Protocol (EXP21D Golden Master):
    1. StandardScaler Amount HANYA di-fit pada train_df, lalu mentransformasikan val_df, test_df, gap_df.
    2. Encoding kategorikal HANYA mempelajari kamus nilai unik dari train_df. Nilai baru pada val/test dipetakan ke 0 (<oov>).
    3. Mengeliminasi Zip dan Merchant City (mode: clean) untuk mencegah pohon keputusan menghafal ID kuasi.
    4. Menambahkan fitur waktu siklik (sin/cos jam & hari) yang bebas memorization.
    5. Menambahkan amount_log (log1p Amount) untuk menstabilkan distribusi numerik.
    """
    print(f"\n[STRICT ANTI-LEAKAGE] Menerapkan preprocessing (Mode: {feature_mode}, Cyclical Time: {use_cyclical_time}, Amount Log: {use_amount_log})...")
    
    # Simpan nilai Amount mentah positif untuk log amount
    for df_split in [train_df, val_df, test_df] + ([gap_df] if gap_df is not None and len(gap_df) > 0 else []):
        if use_amount_log:
            df_split['amount_log'] = np.log1p(np.maximum(0.0, df_split['Amount'].values)).astype(np.float32)

    # Tentukan atribut kategorikal yang diizinkan
    if feature_mode == "clean":
        cat_columns = ['Use Chip', 'Merchant State', 'MCC', 'Errors?']
        print("  • Fitur High-Cardinality ('Zip', 'Merchant City') DIHAPUS dari input!")
    elif feature_mode == "clean_basic":
        cat_columns = ['Use Chip', 'MCC', 'Errors?']
        print(f"  • Menggunakan fitur kategorikal minimal: {cat_columns}")
    else:
        raise ValueError(f"Feature mode '{feature_mode}' tidak dikenal. Pilih 'clean' atau 'clean_basic'.")

    # A. Standarisasi Fitur Numerik (Amount) — FIT HANYA DI TRAIN
    scaler_amount = StandardScaler()
    train_df['Amount'] = scaler_amount.fit_transform(train_df[['Amount']])
    val_df['Amount'] = scaler_amount.transform(val_df[['Amount']])
    test_df['Amount'] = scaler_amount.transform(test_df[['Amount']])
    if gap_df is not None and len(gap_df) > 0:
        gap_df['Amount'] = scaler_amount.transform(gap_df[['Amount']])
        
    print(f"  • Amount Scaler fit pada Train: mean={scaler_amount.mean_[0]:.4f}, std={scaler_amount.scale_[0]:.4f}")

    if use_amount_log:
        scaler_log = StandardScaler()
        train_df['amount_log'] = scaler_log.fit_transform(train_df[['amount_log']])
        val_df['amount_log'] = scaler_log.transform(val_df[['amount_log']])
        test_df['amount_log'] = scaler_log.transform(test_df[['amount_log']])
        if gap_df is not None and len(gap_df) > 0:
            gap_df['amount_log'] = scaler_log.transform(gap_df[['amount_log']])
        print(f"  • Amount Log Scaler fit pada Train: mean={scaler_log.mean_[0]:.4f}, std={scaler_log.scale_[0]:.4f}")

    # B. Robust Categorical Encoding — DICTIONARY FIT HANYA DI TRAIN
    categorical_encoders = {}
    for col in cat_columns:
        train_df[col] = train_df[col].fillna('<missing>').astype(str)
        val_df[col] = val_df[col].fillna('<missing>').astype(str)
        test_df[col] = test_df[col].fillna('<missing>').astype(str)
        if gap_df is not None and len(gap_df) > 0:
            gap_df[col] = gap_df[col].fillna('<missing>').astype(str)
            
        # Pelajari mapping unik dari TRAIN saja
        unique_train_vals = sorted(train_df[col].unique().tolist())
        # Index 0 dicadangkan untuk OOV (<unknown>), kategori train mulai dari index 1
        cat_to_id = {val: idx + 1 for idx, val in enumerate(unique_train_vals)}
        categorical_encoders[col] = cat_to_id
        
        # Transform dengan fallback OOV = 0
        train_df[col] = train_df[col].map(cat_to_id).fillna(0).astype(int)
        val_df[col] = val_df[col].map(cat_to_id).fillna(0).astype(int)
        test_df[col] = test_df[col].map(cat_to_id).fillna(0).astype(int)
        if gap_df is not None and len(gap_df) > 0:
            gap_df[col] = gap_df[col].map(cat_to_id).fillna(0).astype(int)
            
        num_oov_val = int((val_df[col] == 0).sum())
        num_oov_test = int((test_df[col] == 0).sum())
        print(f"  • Kolom '{col}': {len(cat_to_id)} kategori train dipelajari (OOV di Val: {num_oov_val}, di Test: {num_oov_test})")

    # C. Fitur Temporal Siklik (Cyclical Time Features)
    feature_cols = ['Amount']
    if use_amount_log:
        feature_cols.append('amount_log')
    feature_cols = feature_cols + cat_columns

    if use_cyclical_time:
        all_splits = [train_df, val_df, test_df]
        if gap_df is not None and len(gap_df) > 0:
            all_splits.append(gap_df)
            
        for split_df in all_splits:
            hour_float = split_df['datetime'].dt.hour + split_df['datetime'].dt.minute / 60.0
            split_df['hour_sin'] = np.sin(2 * np.pi * hour_float / 24.0).astype(np.float32)
            split_df['hour_cos'] = np.cos(2 * np.pi * hour_float / 24.0).astype(np.float32)
            
            dow = split_df['datetime'].dt.dayofweek
            split_df['dow_sin'] = np.sin(2 * np.pi * dow / 7.0).astype(np.float32)
            split_df['dow_cos'] = np.cos(2 * np.pi * dow / 7.0).astype(np.float32)

        time_cols = ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos']
        feature_cols = feature_cols + time_cols
        print(f"  • Ditambahkan 4 fitur waktu siklik: {time_cols}")

    print(f"Total fitur transaksi bersih (feature_cols): {len(feature_cols)} fitur -> {feature_cols}")
    
    artifacts = {
        "scaler_amount": scaler_amount,
        "categorical_encoders": categorical_encoders,
        "feature_cols": feature_cols,
        "feature_mode": feature_mode
    }
    
    return train_df, val_df, test_df, gap_df, feature_cols, artifacts


# Alias untuk kemudahan kompatibilitas impor
preprocess_features = apply_strict_anti_leakage_preprocessing




In [ ]:
%%writefile graph_builder.py
import torch
import numpy as np
import pandas as pd
from torch_geometric.data import HeteroData
from sklearn.preprocessing import StandardScaler

def graph_level_undersample(df: pd.DataFrame, ratio: float = 1.0, seed: int = 42) -> pd.DataFrame:
    """
    Melakukan Graph-Level Undersampling jika ratio < 1.0.
    Jika ratio == 1.0 (default di exp21d & SHINE), seluruh data dipertahankan
    tanpa merusak topologi asli jaringan.
    """
    if ratio >= 1.0:
        return df
        
    print(f"Menerapkan graph-level undersampling dengan rasio {ratio}...")
    labels = df['Is Fraud?'].values
    fraud_idx = np.flatnonzero(labels == 1)
    normal_idx = np.flatnonzero(labels == 0)
    
    if len(fraud_idx) == 0 or len(normal_idx) == 0:
        return df
        
    desired_normal = int(np.floor(len(normal_idx) * ratio))
    if desired_normal > len(normal_idx):
        desired_normal = len(normal_idx)
        
    rng = np.random.default_rng(seed)
    chosen_normal = rng.choice(normal_idx, size=desired_normal, replace=False)
    
    selected_idx = np.concatenate([chosen_normal, fraud_idx])
    selected_idx.sort()
    
    return df.iloc[selected_idx].reset_index(drop=True)

def build_hetero_graph(df: pd.DataFrame, feature_cols: list, scalers_dict: dict = None) -> tuple:
    """
    Membangun graf heterogen tripartit PyTorch Geometric (HeteroData).
    Nodes:
      - transaction: Fitur transaksi bersih (feature_cols) & label target
      - user: Agregat rata-rata Amount & jumlah transaksi
      - merchant: Agregat rata-rata Amount & jumlah transaksi
    Edges:
      - (user, buys, transaction) & (transaction, rev_buys, user)
      - (merchant, sells, transaction) & (transaction, rev_sells, merchant)

    Anti-Leakage Guard:
    Jika scalers_dict is None, scaler untuk User dan Merchant di-fit pada data saat ini (hanya untuk Train).
    Jika scalers_dict disediakan, scaler dari Train digunakan untuk me-transform User dan Merchant
    pada Validation dan Test tanpa membocorkan statistik masa depan.
    """
    print(f"Membangun HeteroData graf ({len(df)} transaksi)...")
    data = HeteroData()
    
    # 1. Pemetaan ID User dan Merchant ke integer kontigu
    user_mapping = {uid: i for i, uid in enumerate(df['User'].unique())}
    merchant_mapping = {mid: i for i, mid in enumerate(df['Merchant Name'].unique())}
    
    num_users = len(user_mapping)
    num_merchants = len(merchant_mapping)
    num_transactions = len(df)
    
    # 2. Agregasi fitur User dan Merchant
    user_agg = df.groupby('User').agg(
        user_avg_amount=('Amount', 'mean'),
        user_tx_count=('Amount', 'count')
    ).reset_index()
    
    merchant_agg = df.groupby('Merchant Name').agg(
        merchant_avg_amount=('Amount', 'mean'),
        merchant_tx_count=('Amount', 'count')
    ).reset_index()
    
    # Pastikan urutan sesuai dengan mapping
    user_agg['mapped_id'] = user_agg['User'].map(user_mapping)
    user_agg = user_agg.sort_values('mapped_id').reset_index(drop=True)
    
    merchant_agg['mapped_id'] = merchant_agg['Merchant Name'].map(merchant_mapping)
    merchant_agg = merchant_agg.sort_values('mapped_id').reset_index(drop=True)
    
    # Penskalaan fitur User & Merchant (Strict Anti-Leakage)
    if scalers_dict is None:
        scaler_user = StandardScaler()
        user_feats_scaled = scaler_user.fit_transform(user_agg[['user_avg_amount', 'user_tx_count']])
        
        scaler_merchant = StandardScaler()
        merchant_feats_scaled = scaler_merchant.fit_transform(merchant_agg[['merchant_avg_amount', 'merchant_tx_count']])
        
        out_scalers = {
            'scaler_user': scaler_user,
            'scaler_merchant': scaler_merchant
        }
    else:
        scaler_user = scalers_dict['scaler_user']
        scaler_merchant = scalers_dict['scaler_merchant']
        user_feats_scaled = scaler_user.transform(user_agg[['user_avg_amount', 'user_tx_count']])
        merchant_feats_scaled = scaler_merchant.transform(merchant_agg[['merchant_avg_amount', 'merchant_tx_count']])
        out_scalers = scalers_dict
        
    user_features = torch.tensor(user_feats_scaled, dtype=torch.float)
    merchant_features = torch.tensor(merchant_feats_scaled, dtype=torch.float)
    
    # 3. Fitur Transaksi & Label
    tx_features = torch.tensor(df[feature_cols].values, dtype=torch.float)
    tx_labels = torch.tensor(df['Is Fraud?'].values, dtype=torch.float)
    
    data['transaction'].x = tx_features
    data['transaction'].y = tx_labels
    data['transaction'].num_nodes = num_transactions
    
    data['user'].x = user_features
    data['user'].num_nodes = num_users
    
    data['merchant'].x = merchant_features
    data['merchant'].num_nodes = num_merchants
    
    # 4. Membangun Edge Index Bipartit
    tx_indices = np.arange(num_transactions)
    user_src_indices = df['User'].map(user_mapping).values
    merchant_src_indices = df['Merchant Name'].map(merchant_mapping).values
    
    edge_user_to_tx = torch.tensor(np.array([user_src_indices, tx_indices]), dtype=torch.long)
    edge_tx_to_user = torch.tensor(np.array([tx_indices, user_src_indices]), dtype=torch.long)
    
    edge_merchant_to_tx = torch.tensor(np.array([merchant_src_indices, tx_indices]), dtype=torch.long)
    edge_tx_to_merchant = torch.tensor(np.array([tx_indices, merchant_src_indices]), dtype=torch.long)
    
    data['user', 'buys', 'transaction'].edge_index = edge_user_to_tx
    data['transaction', 'rev_buys', 'user'].edge_index = edge_tx_to_user
    
    data['merchant', 'sells', 'transaction'].edge_index = edge_merchant_to_tx
    data['transaction', 'rev_sells', 'merchant'].edge_index = edge_tx_to_merchant
    
    print(f"HeteroData selesai: {num_transactions} tx, {num_users} users, {num_merchants} merchants.")
    return data, out_scalers



In [ ]:
%%writefile model.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, HeteroConv

class HeteroGraphSAGE(nn.Module):
    def __init__(
        self, 
        hidden_channels: int = 128, 
        out_channels: int = 1, 
        num_layers: int = 2, 
        data_metadata: tuple | None = None, 
        feature_dims: dict | None = None, 
        aggr: str = 'mean',
        dropout: float = 0.2,
        edge_types: list | tuple | None = None,
        in_channels_dict: dict | None = None,
        **kwargs,
    ):
        super().__init__()
        
        # Penanganan fleksibel untuk data_metadata & edge_types
        if edge_types is not None:
            resolved_edge_types = list(edge_types)
        elif data_metadata is not None:
            resolved_edge_types = list(data_metadata[1])
        else:
            raise ValueError("Harus menyediakan data_metadata atau edge_types untuk HeteroGraphSAGE.")

        # Penanganan fleksibel untuk feature_dims & in_channels_dict
        resolved_feature_dims = feature_dims if feature_dims is not None else (in_channels_dict or {})
        
        self.num_layers = num_layers
        self.aggr = aggr
        self.dropout_rate = dropout
        self.dropout = nn.Dropout(dropout) if dropout > 0.0 else nn.Identity()
        
        # Inisialisasi proyeksi linear untuk tiap tipe node
        self.lin_embeddings = nn.ModuleDict()
        self.param_embeddings = nn.ParameterDict()
        for node_type, dim in resolved_feature_dims.items():
            if dim == 0:
                self.param_embeddings[node_type] = nn.Parameter(torch.randn(1, hidden_channels))
            else:
                self.lin_embeddings[node_type] = nn.Linear(dim, hidden_channels)
                
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        
        for _ in range(num_layers):
            conv = HeteroConv({
                edge_type: SAGEConv(
                    in_channels=(-1, -1), # Lazy initialization
                    out_channels=hidden_channels,
                    aggr=aggr
                )
                for edge_type in resolved_edge_types
            })
            self.convs.append(conv)
            
            # BatchNorm untuk node transaction
            self.bns.append(nn.BatchNorm1d(hidden_channels))
            
        # Classifier linear bawaan untuk monitoring val AUPRC selama pelatihan GNN
        self.classifier = nn.Linear(hidden_channels, out_channels)

    def forward(self, x_dict, edge_index_dict, num_nodes_dict=None, return_embeddings=False):
        if num_nodes_dict is None:
            num_nodes_dict = {ntype: x.size(0) for ntype, x in x_dict.items()}

        # 1. Proyeksi fitur awal
        h_dict = {}
        
        # Node yang memiliki fitur (transaction, user, merchant)
        for node_type, x in x_dict.items():
            if node_type in self.lin_embeddings:
                h_dict[node_type] = self.lin_embeddings[node_type](x)
            elif node_type in self.param_embeddings:
                num_nodes = num_nodes_dict.get(node_type, 1)
                h_dict[node_type] = self.param_embeddings[node_type].expand(num_nodes, -1)
                
        # 2. Message Passing Layer
        for i in range(self.num_layers):
            h_dict = self.convs[i](h_dict, edge_index_dict)
            if 'transaction' in h_dict and h_dict['transaction'].size(0) > 1:
                h_dict['transaction'] = self.bns[i](h_dict['transaction'])
            h_dict = {k: F.relu(v) for k, v in h_dict.items()}
            h_dict = {k: self.dropout(v) for k, v in h_dict.items()}
            
        tx_emb = h_dict['transaction']
        
        if return_embeddings:
            return tx_emb
            
        out = self.classifier(tx_emb)
        return out.squeeze(-1)

    def get_embeddings(self, x_dict, edge_index_dict, num_nodes_dict=None):
        return self.forward(x_dict, edge_index_dict, num_nodes_dict=num_nodes_dict, return_embeddings=True)

def manual_weighted_bce(logits, targets, pos_weight):
    """
    Weighted Binary Cross Entropy Loss stabil dengan koefisien pos_weight teredam.
    """
    max_val = torch.clamp(-logits, min=0)
    loss = (1 - targets) * logits + (1 + (pos_weight - 1) * targets) * (
        max_val + torch.log(torch.exp(-max_val) + torch.exp(-logits - max_val))
    )
    return loss.mean()

def manual_unweighted_bce(logits, targets):
    """
    Standar Binary Cross Entropy Loss tanpa pembobotan untuk evaluasi objektif.
    """
    max_val = torch.clamp(-logits, min=0)
    loss = (1 - targets) * logits + (
        max_val + torch.log(torch.exp(-max_val) + torch.exp(-logits - max_val))
    )
    return loss.mean()



In [ ]:
%%writefile samplers.py
from __future__ import annotations

from typing import Any
import torch
from torch_geometric.data import HeteroData
from torch_geometric.loader import NeighborLoader


WEIGHT_ATTR = "sampling_weight"
USER_TO_TX = ("user", "buys", "transaction")
TX_TO_USER = ("transaction", "rev_buys", "user")
MERCHANT_TO_TX = ("merchant", "sells", "transaction")
TX_TO_MERCHANT = ("transaction", "rev_sells", "merchant")
WEIGHTED_RELATIONS = (TX_TO_USER, TX_TO_MERCHANT)


def counterpart_source(edge_type: tuple[str, str, str]) -> str:
    if edge_type == TX_TO_USER:
        return "merchant_degree"
    if edge_type == TX_TO_MERCHANT:
        return "user_degree"
    return "not_applicable"


def _bincount(index: torch.Tensor, size: int) -> torch.Tensor:
    return torch.bincount(index, minlength=size).to(dtype=torch.float32)


def _minmax_plus_floor(values: torch.Tensor, alpha: float = 0.1) -> torch.Tensor:
    """Implementasi min-max scaling dengan smoothing floor alpha (Proposal & EXP21D)."""
    if not 0.0 < alpha <= 1.0:
        raise ValueError(f"alpha harus berada pada (0, 1], diterima {alpha}")
    values = values.to(dtype=torch.float32)
    if values.numel() == 0:
        return values
    minimum, maximum = values.min(), values.max()
    if torch.isclose(maximum, minimum):
        return torch.ones_like(values)
    return (values - minimum) / (maximum - minimum + 1e-8) + alpha


def compute_bipartite_jaccard(
    user_edges: torch.Tensor,
    merchant_edges: torch.Tensor,
    num_transactions: int,
    user_deg: torch.Tensor,
    merchant_deg: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Menghitung exact bipartite Jaccard similarity antara User dan Merchant untuk setiap transaksi.
    J(u, m) = |transaksi bersama (u, m)| / (deg(u) + deg(m) - |transaksi bersama (u, m)|)
    Bebas dari label leakage dan bernilai terikat pada rentang [0, 1].
    """
    user_for_tx = user_edges[0]
    merchant_for_tx = merchant_edges[0]

    # Pemetaan pasangan unik (user, merchant) menggunakan 64-bit encoding
    max_m = int(merchant_for_tx.max().item()) + 1 if merchant_for_tx.numel() > 0 else 1
    pair_id = user_for_tx.to(torch.int64) * max_m + merchant_for_tx.to(torch.int64)

    # Menghitung frekuensi kemunculan pasangan (u, m)
    _, inverse_indices, counts = torch.unique(pair_id, return_inverse=True, return_counts=True)
    shared_tx = counts[inverse_indices].to(torch.float32)

    deg_u = user_deg[user_for_tx]
    deg_m = merchant_deg[merchant_for_tx]
    union_deg = (deg_u + deg_m - shared_tx).clamp_min(1.0)

    jaccard_per_tx = (shared_tx / union_deg).clamp(0.0, 1.0)
    return jaccard_per_tx, jaccard_per_tx


def compute_asagnn_attribute_homophily(
    data: HeteroData,
    user_edges: torch.Tensor,
    merchant_edges: torch.Tensor,
    num_transactions: int,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Implementasi Solusi 3: Attribute Homophily berbasis ASA-GNN (Tian et al., IEEE TCSS 2024).
    Mengukur keselarasan atribut transaksi terhadap profil centroid entitas pasangannya:
    H_ASA(x_tx, centroid) = (1 + cos(x_tx, centroid)) / 2
    Menghilangkan ketergantungan pada label L[v] (Zero Label Leakage).
    """
    tx_features = data["transaction"].x.to(torch.float32)
    num_users = int(data["user"].num_nodes)
    num_merchants = int(data["merchant"].num_nodes)
    feat_dim = tx_features.size(1)

    user_for_tx = user_edges[0]
    merchant_for_tx = merchant_edges[0]

    # 1. Centroid fitur per User
    user_feat_sum = torch.zeros((num_users, feat_dim), dtype=torch.float32, device=tx_features.device)
    user_feat_sum.scatter_add_(0, user_for_tx.unsqueeze(1).expand(-1, feat_dim), tx_features)
    user_counts = torch.bincount(user_for_tx, minlength=num_users).to(dtype=torch.float32).clamp_min(1.0).unsqueeze(1)
    user_centroid = user_feat_sum / user_counts

    # 2. Centroid fitur per Merchant
    merchant_feat_sum = torch.zeros((num_merchants, feat_dim), dtype=torch.float32, device=tx_features.device)
    merchant_feat_sum.scatter_add_(0, merchant_for_tx.unsqueeze(1).expand(-1, feat_dim), tx_features)
    merchant_counts = torch.bincount(merchant_for_tx, minlength=num_merchants).to(dtype=torch.float32).clamp_min(1.0).unsqueeze(1)
    merchant_centroid = merchant_feat_sum / merchant_counts

    # 3. Normalized Cosine Similarity
    tx_norm = torch.nn.functional.normalize(tx_features, p=2, dim=1)
    user_norm = torch.nn.functional.normalize(user_centroid, p=2, dim=1)
    merchant_norm = torch.nn.functional.normalize(merchant_centroid, p=2, dim=1)

    # Untuk relasi tx -> user: kandidat dievaluasi terhadap keselarasan atribut merchant pasangannya
    cos_merchant = (tx_norm * merchant_norm[merchant_for_tx]).sum(dim=1).clamp(-1.0, 1.0)
    homophily_for_user_branch = ((1.0 + cos_merchant) / 2.0).clamp(0.0, 1.0)

    # Untuk relasi tx -> merchant: kandidat dievaluasi terhadap keselarasan atribut user pasangannya
    cos_user = (tx_norm * user_norm[user_for_tx]).sum(dim=1).clamp(-1.0, 1.0)
    homophily_for_merchant_branch = ((1.0 + cos_user) / 2.0).clamp(0.0, 1.0)

    return homophily_for_user_branch, homophily_for_merchant_branch


def compute_bipartite_pagerank(
    user_edges: torch.Tensor,
    merchant_edges: torch.Tensor,
    num_users: int,
    num_merchants: int,
    teleport_beta: float = 0.15,
    num_steps: int = 3,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Menghitung skor Personalized PageRank (PPR) lokal/bipartit (Proposal Algoritma 4).
    Menggunakan power iteration terdistribusi sparse matrix multiplication.
    Bebas OOM dan zero temporal leakage.
    """
    total_nodes = num_users + num_merchants
    u_idx = user_edges[0]
    m_idx = merchant_edges[0] + num_users  # Offset indeks merchant

    # Bangun graf bipartit tak-berarah
    src = torch.cat([u_idx, m_idx])
    dst = torch.cat([m_idx, u_idx])
    edge_index = torch.stack([src, dst])

    deg = torch.bincount(edge_index[0], minlength=total_nodes).to(dtype=torch.float32).clamp_min(1.0)
    norm_val = 1.0 / deg[edge_index[0]]

    # Power iteration Personalized PageRank
    pi = torch.ones(total_nodes, dtype=torch.float32) / float(total_nodes)
    teleport = torch.ones(total_nodes, dtype=torch.float32) / float(total_nodes)

    for _ in range(num_steps):
        # Sparse message passing: pi = (1 - beta) * P * pi + beta * e
        incoming = pi[edge_index[0]] * norm_val
        next_pi = torch.zeros(total_nodes, dtype=torch.float32)
        next_pi.scatter_add_(0, edge_index[1], incoming)
        pi = (1.0 - teleport_beta) * next_pi + teleport_beta * teleport

    pi_u = pi[:num_users]
    pi_m = pi[num_users:]

    # Min-max normalisasi skor PPR
    pi_u_norm = (pi_u - pi_u.min()) / (pi_u.max() - pi_u.min() + 1e-8)
    pi_m_norm = (pi_m - pi_m.min()) / (pi_m.max() - pi_m.min() + 1e-8)

    return pi_u_norm, pi_m_norm


def _transaction_topology_structures(data: HeteroData) -> dict[str, Any]:
    """Mengekstrak relasi topologi tripartit heterogen."""
    required = {USER_TO_TX, TX_TO_USER, MERCHANT_TO_TX, TX_TO_MERCHANT}
    missing = required.difference(data.edge_types)
    if missing:
        raise ValueError(f"Relasi graf tripartit belum lengkap: {sorted(missing)}")

    expected = int(data["transaction"].num_nodes)
    user_edges = data[USER_TO_TX].edge_index
    merchant_edges = data[MERCHANT_TO_TX].edge_index

    num_users = int(data["user"].num_nodes)
    num_merchants = int(data["merchant"].num_nodes)

    user_deg = _bincount(user_edges[0], num_users)
    merchant_deg = _bincount(merchant_edges[0], num_merchants)

    user_for_tx = user_edges[0]
    merchant_for_tx = merchant_edges[0]

    return {
        "num_transactions": expected,
        "num_users": num_users,
        "num_merchants": num_merchants,
        "user_edges": user_edges,
        "merchant_edges": merchant_edges,
        "user_deg": user_deg,
        "merchant_deg": merchant_deg,
        "user_for_tx": user_for_tx,
        "merchant_for_tx": merchant_for_tx,
    }


def apply_sampling_weights(
    data: HeteroData,
    method: str,
    alpha: float = 0.1,
    weight_attr: str = WEIGHT_ATTR,
    topology_alpha: float = 0.5,
    importance_gamma: float = 0.3,
    ppr_teleport: float = 0.15,
    ppr_steps: int = 3,
) -> HeteroData:
    """
    Menghitung dan menyematkan bobot sampling pada graf sesuai Proposal EXP24:
    - Uniform: w = 1.0 (Baseline)
    - Topology: w = minmax(alpha * Jaccard + (1 - alpha) * Homophily_ASAGNN) + floor (Algoritma 3 Proposal)
    - Importance: w = minmax(gamma * Degree + (1 - gamma) * PPR) + floor (Algoritma 4 Proposal)
    """
    method = method.lower()
    topo = _transaction_topology_structures(data)
    num_tx = topo["num_transactions"]
    user_edges = topo["user_edges"]
    merchant_edges = topo["merchant_edges"]
    user_deg = topo["user_deg"]
    merchant_deg = topo["merchant_deg"]
    user_for_tx = topo["user_for_tx"]
    merchant_for_tx = topo["merchant_for_tx"]

    # Prekomputasi komponen proposal
    if method == "topology":
        jaccard_u, jaccard_m = compute_bipartite_jaccard(user_edges, merchant_edges, num_tx, user_deg, merchant_deg)
        homo_u, homo_m = compute_asagnn_attribute_homophily(data, user_edges, merchant_edges, num_tx)
        raw_tx_to_user = topology_alpha * jaccard_u + (1.0 - topology_alpha) * homo_u
        raw_tx_to_merchant = topology_alpha * jaccard_m + (1.0 - topology_alpha) * homo_m
    elif method == "importance":
        cd_merchant = merchant_deg / (merchant_deg.max() + 1e-8)
        cd_user = user_deg / (user_deg.max() + 1e-8)
        pi_u, pi_m = compute_bipartite_pagerank(
            user_edges, merchant_edges, topo["num_users"], topo["num_merchants"],
            teleport_beta=ppr_teleport, num_steps=ppr_steps
        )
        raw_tx_to_user = importance_gamma * cd_merchant[merchant_for_tx] + (1.0 - importance_gamma) * pi_m[merchant_for_tx]
        raw_tx_to_merchant = importance_gamma * cd_user[user_for_tx] + (1.0 - importance_gamma) * pi_u[user_for_tx]

    for edge_type in data.edge_types:
        edge_index = data[edge_type].edge_index
        num_edges = int(edge_index.size(1))

        if method == "uniform":
            weights = torch.ones(num_edges, dtype=torch.float32)
            candidate_deg = torch.ones(num_edges, dtype=torch.float32)
        elif edge_type == TX_TO_USER:
            candidate_deg = merchant_deg[merchant_for_tx[edge_index[0]]]
            weights = _minmax_plus_floor(raw_tx_to_user[edge_index[0]], alpha=alpha)
        elif edge_type == TX_TO_MERCHANT:
            candidate_deg = user_deg[user_for_tx[edge_index[0]]]
            weights = _minmax_plus_floor(raw_tx_to_merchant[edge_index[0]], alpha=alpha)
        else:
            # Hop 1 selalu uniform 1.0 (deterministik non-kompetitif)
            weights = torch.ones(num_edges, dtype=torch.float32)
            candidate_deg = torch.ones(num_edges, dtype=torch.float32)

        if weights.numel() != num_edges or not torch.isfinite(weights).all() or (weights <= 0).any():
            raise ValueError(f"Bobot sampling invalid untuk relasi {edge_type}")

        data[edge_type][weight_attr] = weights.contiguous()
        data[edge_type].candidate_degree = candidate_deg.contiguous()

    validate_sampling_weight_contract(data, method, weight_attr)
    return data


def validate_sampling_weight_contract(data: HeteroData, method: str, weight_attr: str = WEIGHT_ATTR) -> None:
    """Verifikasi 8 kriteria integritas sampling contract untuk EXP24."""
    method = method.lower()
    for edge_type in data.edge_types:
        store = data[edge_type]
        if weight_attr not in store:
            raise ValueError(f"{weight_attr} tidak ditemukan pada {edge_type}")
        weights = store[weight_attr]
        destination = store.edge_index[1]
        num_dest_nodes = int(data[edge_type[2]].num_nodes)
        sums = torch.zeros(num_dest_nodes, dtype=torch.float32)
        sums.scatter_add_(0, destination, weights)
        active = torch.bincount(destination, minlength=num_dest_nodes) > 0
        if active.any() and not (sums[active] > 0).all():
            raise ValueError(f"Jumlah bobot lokal non-positif pada {edge_type}")

        # Hop 1 (USER_TO_TX dan MERCHANT_TO_TX) wajib uniform 1.0
        if edge_type not in WEIGHTED_RELATIONS and not torch.allclose(weights, torch.ones_like(weights)):
            raise ValueError(f"Relasi first-hop/non-kompetitif harus bernilai tepat 1.0: {edge_type}")

        if edge_type in WEIGHTED_RELATIONS:
            if method == "uniform" and not torch.allclose(weights, torch.ones_like(weights)):
                raise ValueError(f"Uniform sampler harus berbobot satu pada {edge_type}")
            if method != "uniform" and (weights < 0.1).any():
                raise ValueError(f"Bobot {method} melanggar smoothing floor 0.1 pada {edge_type}")


def sampling_weight_audit(data: HeteroData, method: str, weight_attr: str = WEIGHT_ATTR, **kwargs) -> dict[str, Any]:
    """Menghasilkan laporan audit statistik bobot sampling."""
    # Pastikan bobot sampling sudah diterapkan secara defensif jika belum ada
    for edge_type in data.edge_types:
        if weight_attr not in data[edge_type]:
            apply_sampling_weights(data, method, weight_attr=weight_attr, **kwargs)
            break

    report: dict[str, Any] = {
        "method": method,
        "weight_attr": weight_attr,
        "weighted_relations": ["__".join(item) for item in WEIGHTED_RELATIONS],
        "relations": {},
    }
    for edge_type in data.edge_types:
        store = data[edge_type]
        weights = store[weight_attr].detach().cpu()
        degrees = store.candidate_degree.detach().cpu()
        report["relations"]["__".join(edge_type)] = {
            "num_edges": int(weights.numel()),
            "counterpart_source": counterpart_source(edge_type),
            "weight_min": float(weights.min()) if weights.numel() else None,
            "weight_max": float(weights.max()) if weights.numel() else None,
            "weight_mean": float(weights.mean()) if weights.numel() else None,
            "weight_std": float(weights.std(unbiased=False)) if weights.numel() else None,
            "unique_weights": int(torch.unique(weights).numel()),
            "candidate_degree_min": float(degrees.min()) if degrees.numel() else None,
            "candidate_degree_max": float(degrees.max()) if degrees.numel() else None,
            "contract_passed": True,
        }
    return report


def get_neighbor_loader(
    data: HeteroData,
    method: str = "uniform",
    batch_size: int = 1024,
    shuffle: bool = False,
    num_neighbors: list[int] | None = None,
    seed: int = 42,
    alpha: float = 0.1,
    input_nodes: torch.Tensor | None = None,
    weight_attr: str = WEIGHT_ATTR,
    topology_alpha: float = 0.5,
    importance_gamma: float = 0.3,
    ppr_teleport: float = 0.15,
    ppr_steps: int = 3,
) -> NeighborLoader:
    """Membangun PyG NeighborLoader dengan bobot sampling EXP24 yang telah disesuaikan."""
    if num_neighbors is None:
        num_neighbors = [15, 10]
    if len(num_neighbors) != 2 or any(value <= 0 for value in num_neighbors):
        raise ValueError("num_neighbors harus berisi dua fan-out positif [15, 10]")

    apply_sampling_weights(
        data, method, alpha=alpha, weight_attr=weight_attr,
        topology_alpha=topology_alpha, importance_gamma=importance_gamma,
        ppr_teleport=ppr_teleport, ppr_steps=ppr_steps
    )

    if input_nodes is None:
        input_nodes = torch.arange(int(data["transaction"].num_nodes))

    # PyG NeighborLoader:
    # 1. Untuk method == 'uniform', semua edge memiliki peluang identik (bobot konstan 1.0).
    #    Kita gunakan weight_attr=None agar PyG memakai sampling uniform native yang sangat
    #    cepat dan TIDAK membutuhkan ekstensi C++ eksternal 'pyg_lib'.
    # 2. Untuk method 'topology' dan 'importance', PyG secara ketat membutuhkan 'pyg_lib>=0.3.0'.
    loader_weight_attr = None if method == "uniform" else weight_attr

    if loader_weight_attr is not None:
        try:
            import pyg_lib  # noqa: F401
        except ImportError:
            torch_ver = torch.__version__.split("+")[0]
            cuda_ver = torch.version.cuda
            cuda_tag = f"cu{cuda_ver.replace('.', '')}" if cuda_ver else "cpu"
            wheel_url = f"https://data.pyg.org/whl/torch-{torch_ver}+{cuda_tag}.html"
            raise ImportError(
                f"\n{'='*75}\n"
                f"[EXP24 ERROR] Weighted neighbor sampling (metode '{method}') memerlukan pustaka 'pyg-lib>=0.3.0'.\n"
                f"Pustaka 'pyg_lib' belum terpasang di Python environment Anda.\n\n"
                f"Solusi: Jalankan perintah berikut di Anaconda Prompt pada PC running Anda:\n"
                f"  pip install pyg_lib -f {wheel_url}\n\n"
                f"Atau jika menggunakan conda-forge:\n"
                f"  conda install pyg-lib -c pyg -c conda-forge\n"
                f"{'='*75}\n"
            )

    generator = torch.Generator().manual_seed(seed)
    loader = NeighborLoader(
        data,
        num_neighbors={edge_type: list(num_neighbors) for edge_type in data.edge_types},
        batch_size=batch_size,
        input_nodes=("transaction", input_nodes),
        shuffle=shuffle,
        num_workers=0,
        weight_attr=loader_weight_attr,
        generator=generator,
        subgraph_type="directional",
    )
    return loader


In [ ]:
%%writefile validate_sampler.py
from __future__ import annotations

import json
from types import SimpleNamespace
import torch
from torch_geometric.data import HeteroData

import samplers as sampler_module
from samplers import (
    MERCHANT_TO_TX,
    TX_TO_MERCHANT,
    TX_TO_USER,
    USER_TO_TX,
    apply_sampling_weights,
    get_neighbor_loader,
    sampling_weight_audit,
)


def toy_graph() -> HeteroData:
    """Membangun graf tiruan kecil untuk verifikasi otomatis 8 gate integritas."""
    data = HeteroData()
    torch.manual_seed(42)
    data["transaction"].x = torch.randn(6, 10)
    data["transaction"].y = torch.tensor([0, 1, 0, 0, 1, 0], dtype=torch.float32)
    data["user"].x = torch.randn(3, 2)
    data["merchant"].x = torch.randn(3, 2)
    data["transaction"].num_nodes = 6
    data["user"].num_nodes = 3
    data["merchant"].num_nodes = 3

    transactions = torch.arange(6)
    users = torch.tensor([0, 0, 0, 1, 2, 2])
    merchants = torch.tensor([0, 1, 1, 1, 2, 2])

    data[USER_TO_TX].edge_index = torch.stack([users, transactions])
    data[TX_TO_USER].edge_index = torch.stack([transactions, users])
    data[MERCHANT_TO_TX].edge_index = torch.stack([merchants, transactions])
    data[TX_TO_MERCHANT].edge_index = torch.stack([transactions, merchants])
    return data


def main() -> None:
    print("Menjalankan Correctness Gates untuk EXP24 Proposal Sampler...")

    # 1. Inisialisasi tiga sampler
    uniform = toy_graph()
    topology = toy_graph()
    importance = toy_graph()

    apply_sampling_weights(uniform, "uniform")
    apply_sampling_weights(topology, "topology")
    apply_sampling_weights(importance, "importance")

    # 2. Pengujian Zero Label Leakage
    original_topo_weights = {edge: topology[edge].sampling_weight.clone() for edge in topology.edge_types}
    original_imp_weights = {edge: importance[edge].sampling_weight.clone() for edge in importance.edge_types}

    # Ubah label 100% terbalik dan shuffle acak
    topology["transaction"].y = 1.0 - topology["transaction"].y
    importance["transaction"].y = 1.0 - importance["transaction"].y

    apply_sampling_weights(topology, "topology")
    apply_sampling_weights(importance, "importance")

    topo_leakage_free = all(torch.equal(original_topo_weights[e], topology[e].sampling_weight) for e in topology.edge_types)
    imp_leakage_free = all(torch.equal(original_imp_weights[e], importance[e].sampling_weight) for e in importance.edge_types)

    # 3. Verifikasi Audit
    topology_audit = sampling_weight_audit(topology, "topology")
    importance_audit = sampling_weight_audit(importance, "importance")

    # 4. Verifikasi PyG Loader Kontrak
    real_neighbor_loader = sampler_module.NeighborLoader

    class CaptureNeighborLoader:
        def __init__(self, *args, **kwargs):
            self.node_sampler = SimpleNamespace(weight_attr=kwargs.get("weight_attr"))
            self.kwargs = kwargs

    sampler_module.NeighborLoader = CaptureNeighborLoader
    try:
        loader = get_neighbor_loader(toy_graph(), "topology", batch_size=2, num_neighbors=[2, 2], seed=42)
    finally:
        sampler_module.NeighborLoader = real_neighbor_loader
    active_weight_attr = getattr(loader.node_sampler, "weight_attr", None)

    # 5. Verifikasi Bobot Rentang [0.1, 1.1]
    topo_weights = topology[TX_TO_USER].sampling_weight
    imp_weights = importance[TX_TO_USER].sampling_weight

    checks = {
        "uniform_weights_are_strictly_one": all(
            torch.allclose(uniform[edge].sampling_weight, torch.ones_like(uniform[edge].sampling_weight))
            for edge in uniform.edge_types
        ),
        "first_hop_remains_uniform": (
            torch.allclose(topology[USER_TO_TX].sampling_weight, torch.ones(6))
            and torch.allclose(topology[MERCHANT_TO_TX].sampling_weight, torch.ones(6))
        ),
        "zero_label_leakage_topology": topo_leakage_free,
        "zero_label_leakage_importance": imp_leakage_free,
        "topology_weights_bounded": bool((topo_weights >= 0.1).all() and (topo_weights <= 1.1 + 1e-5).all()),
        "importance_weights_bounded": bool((imp_weights >= 0.1).all() and (imp_weights <= 1.1 + 1e-5).all()),
        "neighbor_loader_uses_weight_attr": active_weight_attr == "sampling_weight",
        "topology_audit_passed": all(item["contract_passed"] for item in topology_audit["relations"].values()),
        "importance_audit_passed": all(item["contract_passed"] for item in importance_audit["relations"].values()),
    }

    report = {"status": "passed" if all(checks.values()) else "failed", "checks": checks}
    print(json.dumps(report, indent=2))
    if report["status"] != "passed":
        raise SystemExit(1)
    print("\n[SUKSES] Seluruh 8 Correctness Gates EXP24 LULUS.")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile main.py
from __future__ import annotations

import argparse
import csv
from datetime import datetime
import json
from pathlib import Path
import shutil
import sys
import time

if sys.platform == "win32":
    try:
        if hasattr(sys.stdout, "reconfigure"):
            sys.stdout.reconfigure(encoding="utf-8", errors="replace")
        if hasattr(sys.stderr, "reconfigure"):
            sys.stderr.reconfigure(encoding="utf-8", errors="replace")
    except Exception:
        pass
import numpy as np
import psutil
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
import torch
from xgboost import XGBClassifier

from artifacts import append_event, write_json
from config import load_config
from data_preprocessing import apply_strict_anti_leakage_preprocessing, load_and_split_raw_data
from graph_builder import build_hetero_graph
from model import HeteroGraphSAGE
from samplers import apply_sampling_weights, get_neighbor_loader, sampling_weight_audit


class DualLogger:
    """Menduplikasikan output stdout & stderr secara real-time ke terminal dan ke berkas log."""
    def __init__(self, log_path: Path):
        self.terminal_stdout = sys.stdout
        self.terminal_stderr = sys.stderr
        log_path.parent.mkdir(parents=True, exist_ok=True)
        self.log_file = open(log_path, "a", encoding="utf-8")
        self._closed = False
        sys.stdout = self
        sys.stderr = self

    def write(self, message: str) -> None:
        try:
            self.terminal_stdout.write(message)
        except UnicodeEncodeError:
            encoding = getattr(self.terminal_stdout, "encoding", "utf-8") or "utf-8"
            self.terminal_stdout.write(message.encode(encoding, errors="replace").decode(encoding))
        self.terminal_stdout.flush()
        if not self._closed:
            try:
                self.log_file.write(message)
                self.log_file.flush()
            except Exception:
                pass

    def flush(self) -> None:
        self.terminal_stdout.flush()
        if not self._closed:
            try:
                self.log_file.flush()
            except Exception:
                pass

    def close(self) -> None:
        if not self._closed:
            self._closed = True
            sys.stdout = self.terminal_stdout
            sys.stderr = self.terminal_stderr
            try:
                self.log_file.close()
            except Exception:
                pass


def update_runs_index(index_path: Path, entry: dict) -> None:
    """Mencatat entri run ke dalam runs_index.csv untuk pemantauan historis terpusat."""
    index_path.parent.mkdir(parents=True, exist_ok=True)
    headers = [
        "run_id", "timestamp", "sampling_method", "seed", "nrows",
        "best_epoch", "val_auprc", "test_auprc", "test_f1",
        "optimal_threshold", "test_fp", "test_fn", "runtime_sec",
        "status", "run_dir", "log_file"
    ]
    file_exists = index_path.is_file()
    with index_path.open("a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=headers)
        if not file_exists:
            writer.writeheader()
        writer.writerow({k: entry.get(k, "") for k in headers})


def manual_weighted_bce(logits: torch.Tensor, targets: torch.Tensor, pos_weight: float) -> torch.Tensor:
    probs = torch.sigmoid(logits)
    probs = torch.clamp(probs, min=1e-7, max=1.0 - 1e-7)
    weights = torch.where(targets == 1.0, torch.tensor(pos_weight, device=logits.device), torch.tensor(1.0, device=logits.device))
    loss = -(weights * (targets * torch.log(probs) + (1.0 - targets) * torch.log(1.0 - probs)))
    return loss.mean()


def manual_unweighted_bce(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    probs = torch.sigmoid(logits)
    probs = torch.clamp(probs, min=1e-7, max=1.0 - 1e-7)
    loss = -(targets * torch.log(probs) + (1.0 - targets) * torch.log(1.0 - probs))
    return loss.mean()


def find_optimal_threshold(y_probs: np.ndarray, y_true: np.ndarray, min_precision: float = 0.15) -> tuple[float, dict]:
    # Edge case: jika tidak ada sampel positif sama sekali (misal pada smoke test nrows kecil)
    if (y_true == 1).sum() == 0:
        optimal_threshold = 0.5
        y_pred = (y_probs >= optimal_threshold).astype(int)
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        tn, fp, fn, tp = [int(v) for v in cm.ravel()]
        return optimal_threshold, {
            "optimal_threshold": 0.5,
            "f1_score": 0.0,
            "precision": 0.0,
            "recall": 0.0,
            "tn": tn,
            "fp": fp,
            "fn": fn,
            "tp": tp,
        }

    precisions, recalls, thresholds = precision_recall_curve(y_true, y_probs)
    f1_scores = np.divide(
        2 * (precisions * recalls),
        (precisions + recalls),
        out=np.zeros_like(precisions),
        where=(precisions + recalls) > 0,
    )
    valid_mask = precisions[:-1] >= min_precision
    if np.any(valid_mask):
        best_idx = np.argmax(f1_scores[:-1][valid_mask])
        optimal_threshold = float(thresholds[valid_mask][best_idx])
    elif len(f1_scores) > 1 and len(thresholds) > 0:
        best_idx = np.argmax(f1_scores[:-1])
        optimal_threshold = float(thresholds[best_idx])
    else:
        optimal_threshold = 0.5

    y_pred = (y_probs >= optimal_threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = [int(v) for v in cm.ravel()]

    metrics = {
        "optimal_threshold": float(optimal_threshold),
        "f1_score": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    }
    return optimal_threshold, metrics


def evaluate_predictions(y_probs: np.ndarray, y_true: np.ndarray, optimal_tau: float) -> dict:
    if (y_true == 1).sum() == 0 or (y_true == 0).sum() == 0:
        auprc = 0.0
        roc_auc = 0.5
    else:
        try:
            auprc = float(average_precision_score(y_true, y_probs))
        except Exception:
            auprc = 0.0
        try:
            roc_auc = float(roc_auc_score(y_true, y_probs))
        except Exception:
            roc_auc = 0.5

    y_pred_def = (y_probs >= 0.5).astype(int)
    cm_def = confusion_matrix(y_true, y_pred_def, labels=[0, 1])
    tn_d, fp_d, fn_d, tp_d = [int(v) for v in cm_def.ravel()]
    metrics_def = {
        "threshold": 0.5,
        "f1_score": float(f1_score(y_true, y_pred_def, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred_def, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred_def, zero_division=0)),
        "tn": tn_d,
        "fp": fp_d,
        "fn": fn_d,
        "tp": tp_d,
    }

    y_pred_opt = (y_probs >= optimal_tau).astype(int)
    cm_opt = confusion_matrix(y_true, y_pred_opt, labels=[0, 1])
    tn_o, fp_o, fn_o, tp_o = [int(v) for v in cm_opt.ravel()]
    metrics_opt = {
        "threshold": float(optimal_tau),
        "f1_score": float(f1_score(y_true, y_pred_opt, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred_opt, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred_opt, zero_division=0)),
        "tn": tn_o,
        "fp": fp_o,
        "fn": fn_o,
        "tp": tp_o,
    }

    return {
        "metrics_threshold_independent": {"auprc": auprc, "roc_auc": roc_auc},
        "metrics_default_threshold_05": metrics_def,
        "metrics_optimal_threshold": metrics_opt,
    }


def extract_features(model, loader, device, num_nodes, feat_dim_tab, hidden_dim):
    model.eval()
    feat_matrix = np.zeros((num_nodes, feat_dim_tab + hidden_dim), dtype=np.float32)
    labels = np.zeros(num_nodes, dtype=np.float32)
    curr_idx = 0
    start_time = time.time()
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            num_nodes_dict = {ntype: batch[ntype].num_nodes for ntype in batch.node_types}
            embs = model.get_embeddings(batch.x_dict, batch.edge_index_dict, num_nodes_dict=num_nodes_dict)
            bs = batch["transaction"].batch_size
            next_idx = min(curr_idx + bs, num_nodes)
            actual_count = next_idx - curr_idx
            if actual_count <= 0:
                break
            embs_cur = embs[:actual_count].cpu().numpy()
            tab_cur = batch["transaction"].x[:actual_count].cpu().numpy()
            y_cur = batch["transaction"].y[:actual_count].cpu().numpy()
            feat_matrix[curr_idx : next_idx] = np.hstack([tab_cur, embs_cur])
            labels[curr_idx : next_idx] = y_cur
            curr_idx = next_idx
    return feat_matrix, labels, time.time() - start_time


def main() -> None:
    parser = argparse.ArgumentParser(description="EXP24 Proposal-Aligned Neighbor Sampler Training")
    parser.add_argument("--config", type=str, default="config.reference.yaml")
    parser.add_argument("--sampling", type=str, default=None)
    parser.add_argument("--seed", type=int, default=None)
    parser.add_argument("--device", type=str, default=None)
    parser.add_argument("--nrows", type=int, default=None)
    parser.add_argument("--run-id", type=str, default=None, help="ID unik untuk run ini (default: auto timestamped)")
    parser.add_argument("--log-to-file", action="store_true", default=None, help="Paksa aktifkan pencatatan log ke file")
    parser.add_argument("--no-log-to-file", action="store_true", help="Nonaktifkan pencatatan log ke file")
    args = parser.parse_args()

    cfg = load_config(args.config)
    exp_cfg = cfg.get("experiment", {})
    t_cfg = cfg.get("training", {})
    s_cfg = cfg.get("sampling", {})
    tw_cfg = cfg.get("time_window", {})
    out_cfg = cfg.get("output", {})
    feat_cfg = cfg.get("features", {})
    log_cfg = cfg.get("logging", {})

    seed = args.seed if args.seed is not None else exp_cfg.get("seed", 42)
    evaluation_seed = exp_cfg.get("evaluation_seed", seed + 1000)
    sampling_method = (args.sampling or s_cfg.get("method", "uniform")).lower()

    if sampling_method not in ["uniform", "topology", "importance"]:
        raise ValueError(f"Metode sampling {sampling_method} tidak valid. Pilih uniform, topology, atau importance.")

    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    device_str = args.device or ("cuda" if torch.cuda.is_available() else "cpu")
    device = torch.device(device_str)

    # 1. Tentukan Run ID
    timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    nrows_tag = f"_nr{args.nrows}" if args.nrows is not None else ""
    default_run_id = f"exp24_{sampling_method}_s{seed}{nrows_tag}_{timestamp_str}"
    run_id = args.run_id or exp_cfg.get("run_id") or default_run_id

    # 2. Tentukan Direktori Output
    result_root = Path(out_cfg.get("result_dir", "../../result/exp24_proposal_sampler"))
    runs_root = Path(out_cfg.get("runs_dir", str(result_root / "runs")))
    model_root = Path(out_cfg.get("model_dir", "../../model/exp24_proposal_sampler"))
    log_root = Path(log_cfg.get("log_dir", str(result_root / "logs")))

    # Folder terisolasi khusus untuk run ini (mencegah hasil tercampur)
    isolate_runs = out_cfg.get("isolate_runs", True)
    if isolate_runs:
        result_dir = runs_root / run_id
        model_dir = model_root / "runs" / run_id
    else:
        result_dir = result_root / f"seed_{seed}"
        model_dir = model_root / f"seed_{seed}"

    result_dir.mkdir(parents=True, exist_ok=True)
    model_dir.mkdir(parents=True, exist_ok=True)

    # Direktori canonical seed (untuk kompatibilitas multi-seed aggregasi EXP24)
    canonical_seed_dir = result_root / f"seed_{seed}"
    canonical_seed_dir.mkdir(parents=True, exist_ok=True)
    canonical_model_dir = model_root / f"seed_{seed}"
    canonical_model_dir.mkdir(parents=True, exist_ok=True)

    # 3. Setup File Logging jika diaktifkan di YAML atau CLI
    should_log = log_cfg.get("log_to_file", True)
    if args.no_log_to_file:
        should_log = False
    elif args.log_to_file:
        should_log = True

    logger = None
    log_path = None
    if should_log:
        log_root.mkdir(parents=True, exist_ok=True)
        log_path = log_root / f"{run_id}.log"
        logger = DualLogger(log_path)
        import atexit
        atexit.register(logger.close)

    start_total_time = time.time()
    events_path = result_dir / "events.jsonl"
    csv_history_path = result_dir / f"exp24_{sampling_method}_training_history.csv"
    json_history_path = result_dir / f"exp24_{sampling_method}_history.json"

    print("=" * 70)
    print("  EXP24: PROPOSAL-ALIGNED NEIGHBOR SAMPLING FOR GRAPHSAGE")
    print(f"  Run ID  : {run_id}")
    print(f"  Sampler : {sampling_method.upper()} | Seed: {seed} | Device: {device_str}")
    print(f"  Run Dir : {result_dir}")
    if log_path:
        print(f"  Log File: {log_path}")
    print("=" * 70)

    append_event(events_path, "run_initialized", run_id=run_id, sampling_method=sampling_method, seed=seed)

    # 1. Pemuatan & Pembagian Temporal
    t_prep_start = time.time()
    data_path = cfg.get("data", {}).get("transactions")
    nrows = args.nrows or cfg.get("data", {}).get("max_rows", None)
    train_df, val_df, test_df, gap_df, stats_dict = load_and_split_raw_data(
        file_path=data_path,
        start_year=cfg.get("data", {}).get("start_year", 2016),
        nrows=nrows,
        reference_date=tw_cfg.get("reference_date", "2018-01-01"),
        train_days=tw_cfg.get("train_days", 300),
        val_days=tw_cfg.get("val_days", 30),
        gap_days=tw_cfg.get("gap_days", 8),
        test_days=tw_cfg.get("test_days", 60),
    )

    # 2. Preprocessing Fitur Tabular (Clean 10-d)
    train_df, val_df, test_df, gap_df, feature_cols, art_prep = apply_strict_anti_leakage_preprocessing(
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        gap_df=gap_df,
        feature_mode=feat_cfg.get("mode", "clean"),
        use_cyclical_time=feat_cfg.get("use_cyclical_time", True),
        use_amount_log=feat_cfg.get("use_amount_log", True),
    )

    # 3. Konstruksi Graf HeteroData
    train_graph, scalers = build_hetero_graph(train_df, feature_cols, scalers_dict=None)
    val_graph, _ = build_hetero_graph(val_df, feature_cols, scalers_dict=scalers)
    test_graph, _ = build_hetero_graph(test_df, feature_cols, scalers_dict=scalers)
    data_prep_time = time.time() - t_prep_start

    # 4. Audit & Penerapan Bobot Sampler Proposal
    alpha = s_cfg.get("smoothing_alpha", 0.1)
    weight_attr = s_cfg.get("weight_attr", "sampling_weight")
    topo_alpha = s_cfg.get("topology_alpha", 0.5)
    imp_gamma = s_cfg.get("importance_gamma", 0.3)
    ppr_teleport = s_cfg.get("ppr_teleport", 0.15)
    ppr_steps = s_cfg.get("ppr_steps", 3)

    for g in (train_graph, val_graph, test_graph):
        apply_sampling_weights(
            g,
            method=sampling_method,
            alpha=alpha,
            weight_attr=weight_attr,
            topology_alpha=topo_alpha,
            importance_gamma=imp_gamma,
            ppr_teleport=ppr_teleport,
            ppr_steps=ppr_steps,
        )

    audit_report = sampling_weight_audit(train_graph, sampling_method, weight_attr=weight_attr)
    audit_path = result_dir / f"exp24_{sampling_method}_sampler_weight_audit.json"
    write_json(audit_path, audit_report)

    # 5. Inisialisasi Loader PyG
    batch_size = t_cfg.get("batch_size", 1024)
    num_neighbors = t_cfg.get("num_neighbors", [15, 10])
    train_loader = get_neighbor_loader(
        train_graph,
        method=sampling_method,
        batch_size=batch_size,
        shuffle=True,
        num_neighbors=num_neighbors,
        seed=seed,
        alpha=alpha,
        weight_attr=weight_attr,
        topology_alpha=s_cfg.get("topology_alpha", 0.5),
        importance_gamma=s_cfg.get("importance_gamma", 0.3),
        ppr_teleport=s_cfg.get("ppr_teleport", 0.15),
        ppr_steps=s_cfg.get("ppr_steps", 3),
    )
    val_loader = get_neighbor_loader(
        val_graph,
        method=sampling_method,
        batch_size=batch_size,
        shuffle=False,
        num_neighbors=num_neighbors,
        seed=evaluation_seed,
        alpha=alpha,
        weight_attr=weight_attr,
    )
    test_loader = get_neighbor_loader(
        test_graph,
        method=sampling_method,
        batch_size=batch_size,
        shuffle=False,
        num_neighbors=num_neighbors,
        seed=evaluation_seed + 1,
        alpha=alpha,
        weight_attr=weight_attr,
    )

    # 6. Model HeteroGraphSAGE
    hidden_dim = cfg.get("model", {}).get("hidden_channels", 128)
    num_layers = cfg.get("model", {}).get("num_layers", 2)
    aggr = cfg.get("model", {}).get("aggr", "mean")
    dropout = cfg.get("model", {}).get("dropout", 0.2)

    feature_dims = {
        "transaction": len(feature_cols),
        "user": train_graph["user"].x.size(1),
        "merchant": train_graph["merchant"].x.size(1),
    }
    model = HeteroGraphSAGE(
        hidden_channels=hidden_dim,
        out_channels=1,
        num_layers=num_layers,
        data_metadata=train_graph.metadata(),
        feature_dims=feature_dims,
        edge_types=train_graph.edge_types,
        aggr=aggr,
        dropout=dropout,
    ).to(device)

    # Inisialisasi parameter lazy layer via dry run 1 mini-batch sebelum Optimizer
    for dry_batch in train_loader:
        dry_batch = dry_batch.to(device)
        dry_num_nodes = {ntype: dry_batch[ntype].num_nodes for ntype in dry_batch.node_types}
        _ = model(dry_batch.x_dict, dry_batch.edge_index_dict, num_nodes_dict=dry_num_nodes)
        break

    # Loss & Pos Weight
    num_pos = float((train_df["Is Fraud?"] == 1).sum())
    num_neg = float((train_df["Is Fraud?"] == 0).sum())
    sqrt_pos_weight = float(np.clip(np.sqrt(num_neg / (num_pos + 1e-8)), 1.0, t_cfg.get("max_pos_weight", 50.0)))

    optimizer = torch.optim.Adam(model.parameters(), lr=t_cfg.get("learning_rate", 0.001), weight_decay=t_cfg.get("weight_decay", 1e-4))
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=t_cfg.get("lr_scheduler_factor", 0.5), patience=t_cfg.get("lr_scheduler_patience", 3))

    # 7. Inisialisasi Logging CSV & JSON Detail Per-Epoch
    csv_headers = [
        "epoch", "train_loss", "train_acc", "val_loss", "val_auprc", "val_roc_auc",
        "val_f1_default", "val_f1_optimal", "val_precision_optimal", "val_recall_optimal",
        "val_optimal_threshold", "val_fp", "val_fn", "learning_rate",
        "time_train_epoch_sec", "time_val_epoch_sec", "time_epoch_total_sec",
        "cumulative_runtime_sec", "val_latency_ms_per_1k_tx",
        "peak_gpu_vram_mb", "peak_ram_mb", "is_best_epoch"
    ]
    with csv_history_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(csv_headers)

    training_history = {k: [] for k in csv_headers}
    epochs = t_cfg.get("epochs", 20)
    early_stopping_patience = t_cfg.get("early_stopping_patience", 7)
    best_val_auprc = 0.0
    best_epoch = 0
    patience_counter = 0
    best_model_state = None
    cumulative_time = 0.0

    print("\n" + "=" * 70)
    print("  MEMULAI PELATIHAN HETEROGRAPHSAGE DENGAN TELEMETRI GRANULAR")
    print("=" * 70)

    for epoch in range(1, epochs + 1):
        t_epoch_start = time.perf_counter()

        # Phase A: Training Pass
        model.train()
        total_loss = 0.0
        total_correct = 0
        total_tx = 0
        t_train_start = time.perf_counter()

        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            num_nodes_dict = {ntype: batch[ntype].num_nodes for ntype in batch.node_types}
            out = model(batch.x_dict, batch.edge_index_dict, num_nodes_dict=num_nodes_dict)

            bs = batch["transaction"].batch_size
            logits = out[:bs]
            targets = batch["transaction"].y[:bs]

            loss = manual_weighted_bce(logits, targets, sqrt_pos_weight)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), t_cfg.get("grad_clip_norm", 5.0))
            optimizer.step()

            total_loss += float(loss.item()) * bs
            preds = (torch.sigmoid(logits) >= 0.5).float()
            total_correct += int((preds == targets).sum().item())
            total_tx += bs

        time_train_epoch_sec = time.perf_counter() - t_train_start
        train_loss = total_loss / total_tx if total_tx > 0 else 0.0
        train_acc = total_correct / total_tx if total_tx > 0 else 0.0

        # Phase B: Validation Evaluation Pass
        model.eval()
        t_val_start = time.perf_counter()
        val_loss_total = 0.0
        val_tx_count = 0
        all_val_probs = []
        all_val_targets = []

        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                num_nodes_dict = {ntype: batch[ntype].num_nodes for ntype in batch.node_types}
                out = model(batch.x_dict, batch.edge_index_dict, num_nodes_dict=num_nodes_dict)

                bs = batch["transaction"].batch_size
                logits = out[:bs]
                targets = batch["transaction"].y[:bs]

                loss_v = manual_unweighted_bce(logits, targets)
                val_loss_total += float(loss_v.item()) * bs
                val_tx_count += bs

                probs = torch.sigmoid(logits).cpu().numpy()
                all_val_probs.append(probs)
                all_val_targets.append(targets.cpu().numpy())

        time_val_epoch_sec = time.perf_counter() - t_val_start
        val_loss = val_loss_total / val_tx_count if val_tx_count > 0 else 0.0
        val_probs_arr = np.concatenate(all_val_probs)
        val_targets_arr = np.concatenate(all_val_targets)

        # Metrik threshold-independent
        if (val_targets_arr == 1).sum() == 0 or (val_targets_arr == 0).sum() == 0:
            val_auprc = 0.0
            val_roc_auc = 0.5
        else:
            try:
                val_auprc = float(average_precision_score(val_targets_arr, val_probs_arr))
            except Exception:
                val_auprc = 0.0
            try:
                val_roc_auc = float(roc_auc_score(val_targets_arr, val_probs_arr))
            except Exception:
                val_roc_auc = 0.5

        # Metrik threshold 0.5
        val_preds_05 = (val_probs_arr >= 0.5).astype(int)
        val_f1_default = float(f1_score(val_targets_arr, val_preds_05, zero_division=0))

        # Kalibrasi threshold optimal PR-Curve pada validasi
        opt_tau, val_metrics_opt = find_optimal_threshold(val_probs_arr, val_targets_arr, min_precision=0.15)
        val_f1_opt = val_metrics_opt["f1_score"]
        val_prec_opt = val_metrics_opt["precision"]
        val_rec_opt = val_metrics_opt["recall"]
        val_fp = val_metrics_opt["fp"]
        val_fn = val_metrics_opt["fn"]

        # Waktu & Sumber Daya
        time_epoch_total_sec = time.perf_counter() - t_epoch_start
        cumulative_time += time_epoch_total_sec
        val_latency_ms_per_1k = (time_val_epoch_sec / max(val_tx_count, 1)) * 1000.0 * 1000.0

        peak_vram_mb = float(torch.cuda.max_memory_allocated() / (1024**2)) if torch.cuda.is_available() else 0.0
        peak_ram_mb = float(psutil.Process().memory_info().rss / (1024**2))

        # Update Scheduler
        if scheduler is not None:
            scheduler.step(val_auprc)
        current_lr = float(optimizer.param_groups[0]["lr"])

        is_best_epoch = bool(val_auprc > best_val_auprc)
        if is_best_epoch:
            best_val_auprc = val_auprc
            best_epoch = epoch
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        # Simpan baris CSV telemetri
        row_dict = {
            "epoch": epoch,
            "train_loss": round(train_loss, 5),
            "train_acc": round(train_acc, 4),
            "val_loss": round(val_loss, 5),
            "val_auprc": round(val_auprc, 5),
            "val_roc_auc": round(val_roc_auc, 5),
            "val_f1_default": round(val_f1_default, 4),
            "val_f1_optimal": round(val_f1_opt, 4),
            "val_precision_optimal": round(val_prec_opt, 4),
            "val_recall_optimal": round(val_rec_opt, 4),
            "val_optimal_threshold": round(opt_tau, 4),
            "val_fp": val_fp,
            "val_fn": val_fn,
            "learning_rate": current_lr,
            "time_train_epoch_sec": round(time_train_epoch_sec, 2),
            "time_val_epoch_sec": round(time_val_epoch_sec, 2),
            "time_epoch_total_sec": round(time_epoch_total_sec, 2),
            "cumulative_runtime_sec": round(cumulative_time, 2),
            "val_latency_ms_per_1k_tx": round(val_latency_ms_per_1k, 2),
            "peak_gpu_vram_mb": round(peak_vram_mb, 2),
            "peak_ram_mb": round(peak_ram_mb, 2),
            "is_best_epoch": is_best_epoch,
        }

        with csv_history_path.open("a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow([row_dict[k] for k in csv_headers])

        for k in csv_headers:
            training_history[k].append(row_dict[k])
        write_json(json_history_path, training_history)

        append_event(
            events_path,
            "epoch_completed",
            method=sampling_method,
            epoch=epoch,
            train_loss=train_loss,
            val_loss=val_loss,
            val_auprc=val_auprc,
            val_f1=val_f1_opt,
            time_epoch_sec=time_epoch_total_sec,
            peak_vram_mb=peak_vram_mb,
            is_best=is_best_epoch,
        )

        star = " [BEST]" if is_best_epoch else ""
        print(
            f"Epoch {epoch:02d}/{epochs:02d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val AUPRC: {val_auprc:.4f} | "
            f"Val F1 (Opt): {val_f1_opt:.4f} (Tau: {opt_tau:.3f}) | "
            f"Waktu: {time_epoch_total_sec:.1f}s (Train: {time_train_epoch_sec:.1f}s, Val: {time_val_epoch_sec:.1f}s){star}"
        )

        if patience_counter >= early_stopping_patience:
            print(f"[EARLY STOPPING] Tidak ada peningkatan AUPRC selama {early_stopping_patience} epoch berturut-turut. Berhenti di epoch {epoch}.")
            append_event(events_path, "early_stopping_triggered", epoch=epoch, best_epoch=best_epoch, best_auprc=best_val_auprc)
            break

    # Pulihkan checkpoint terbaik
    if best_model_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_model_state.items()})

    best_model_path = model_dir / f"exp24_{sampling_method}_best_gnn.pt"
    torch.save(model.state_dict(), best_model_path)
    print(f"\n[CHECKPOINT] Model GNN terbaik disimpan ke: {best_model_path} (AUPRC: {best_val_auprc:.4f} pada Epoch {best_epoch})")

    # 8. Ekstraksi Fitur Hibrida (10-d Tabular + 128-d GNN Embedding)
    print("\n" + "=" * 70)
    print("  EKSTRAKSI FITUR HIBRIDA (10-d TABULAR + 128-d GNN EMBEDDING)")
    print("=" * 70)
    feat_dim_tab = len(feature_cols)
    train_extract_loader = get_neighbor_loader(
        train_graph, method=sampling_method, batch_size=batch_size, shuffle=False,
        num_neighbors=num_neighbors, seed=evaluation_seed + 2, alpha=alpha, weight_attr=weight_attr
    )
    X_train, y_train, inf_train_t = extract_features(model, train_extract_loader, device, len(train_df), feat_dim_tab, hidden_dim)
    X_val, y_val, inf_val_t = extract_features(model, val_loader, device, len(val_df), feat_dim_tab, hidden_dim)
    X_test, y_test, inf_test_t = extract_features(model, test_loader, device, len(test_df), feat_dim_tab, hidden_dim)

    # 9. Pelatihan Downstream XGBoost Classifier
    print("\n" + "=" * 70)
    print("  PELATIHAN DOWNSTREAM HYBRID XGBOOST")
    print("=" * 70)
    c_cfg = cfg.get("classifier", {})
    xgb_params = {
        "n_estimators": c_cfg.get("n_estimators", 100),
        "max_depth": c_cfg.get("max_depth", 4),
        "learning_rate": c_cfg.get("learning_rate", 0.05),
        "min_child_weight": c_cfg.get("min_child_weight", 10),
        "subsample": c_cfg.get("subsample", 0.8),
        "colsample_bytree": c_cfg.get("colsample_bytree", 0.8),
        "scale_pos_weight": sqrt_pos_weight,
        "random_state": seed,
        "n_jobs": -1,
        "eval_metric": "logloss",
        "tree_method": c_cfg.get("tree_method", "hist"),
    }

    t_xgb_start = time.perf_counter()
    xgb_hybrid = XGBClassifier(**xgb_params)
    xgb_hybrid.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    xgb_train_time = time.perf_counter() - t_xgb_start

    # Simpan riwayat boosting XGBoost
    xgb_history_path = result_dir / f"exp24_{sampling_method}_xgboost_history.json"
    write_json(xgb_history_path, {"hybrid": xgb_hybrid.evals_result(), "fit_time_sec": xgb_train_time})

    hybrid_val_probs = xgb_hybrid.predict_proba(X_val)[:, 1]
    hybrid_test_probs = xgb_hybrid.predict_proba(X_test)[:, 1]

    # Kalibrasi threshold HANYA dari set validasi (Strict Anti-Leakage)
    optimal_tau, tau_metrics = find_optimal_threshold(hybrid_val_probs, y_val, min_precision=0.15)
    test_eval_results = evaluate_predictions(hybrid_test_probs, y_test, optimal_tau=optimal_tau)
    m_opt = test_eval_results["metrics_optimal_threshold"]
    m_ind = test_eval_results["metrics_threshold_independent"]

    # Simpan Prediksi Lengkap NPZ
    predictions_path = result_dir / f"exp24_{sampling_method}_predictions.npz"
    np.savez_compressed(
        predictions_path,
        hybrid_test_probabilities=hybrid_test_probs.astype(np.float32),
        hybrid_val_probabilities=hybrid_val_probs.astype(np.float32),
        test_labels=y_test.astype(np.uint8),
        val_labels=y_val.astype(np.uint8),
        hybrid_optimal_threshold=float(optimal_tau),
    )

    # 10. Penyimpanan Hasil Akhir Manifest & Results JSON
    total_pipeline_time = time.time() - start_total_time
    results_summary = {
        "experiment_id": "EXP24-ProposalSampler",
        "sampling_method": sampling_method,
        "seed": seed,
        "best_epoch": best_epoch,
        "best_val_auprc": float(best_val_auprc),
        "test_metrics": {
            "auprc": m_ind["auprc"],
            "roc_auc": m_ind["roc_auc"],
            "f1_score": m_opt["f1_score"],
            "precision": m_opt["precision"],
            "recall": m_opt["recall"],
            "optimal_threshold": float(optimal_tau),
            "fp": m_opt["fp"],
            "fn": m_opt["fn"],
        },
        "runtime_telemetry": {
            "data_prep_sec": data_prep_time,
            "gnn_training_sec": cumulative_time,
            "xgb_training_sec": xgb_train_time,
            "test_inference_sec": inf_test_t,
            "total_pipeline_sec": total_pipeline_time,
            "val_latency_ms_per_1k": (inf_val_t / len(val_df)) * 1000.0 * 1000.0,
        },
    }
    results_summary["run_id"] = run_id
    results_summary["is_smoke_test"] = args.nrows is not None
    results_summary["nrows"] = args.nrows
    write_json(result_dir / f"exp24_{sampling_method}_results.json", results_summary)

    manifest = {
        "experiment_id": "EXP24-ProposalSampler",
        "run_id": run_id,
        "timestamp": datetime.now().isoformat(),
        "status": "complete",
        "seed": seed,
        "evaluation_seed": evaluation_seed,
        "sampling_method": sampling_method,
        "paired_block": f"seed_{seed}",
        "nrows": args.nrows,
        "is_smoke_test": args.nrows is not None,
        "predictions_file": str(predictions_path.name),
        "history_csv": str(csv_history_path.name),
        "history_json": str(json_history_path.name),
        "events_log": str(events_path.name),
        "log_file": str(log_path.resolve()) if log_path else None,
        "run_dir": str(result_dir.resolve()),
    }
    write_json(result_dir / f"exp24_{sampling_method}_manifest.json", manifest)

    # Selalu sinkronisasikan artefak terbaru ke canonical seed directory (seed_{seed})
    # agar modul orchestrator (run_seed.py) dan agregasi (aggregate_exp24.py) selalu dapat menemukannya
    for fname in [
        f"exp24_{sampling_method}_predictions.npz",
        f"exp24_{sampling_method}_results.json",
        f"exp24_{sampling_method}_manifest.json",
        f"exp24_{sampling_method}_training_history.csv",
        f"exp24_{sampling_method}_history.json",
        f"exp24_{sampling_method}_xgboost_history.json",
        f"exp24_{sampling_method}_sampler_weight_audit.json",
    ]:
        src = result_dir / fname
        if src.is_file():
            shutil.copy2(src, canonical_seed_dir / fname)
    src_gnn = model_dir / f"exp24_{sampling_method}_best_gnn.pt"
    if src_gnn.is_file():
        shutil.copy2(src_gnn, canonical_model_dir / f"exp24_{sampling_method}_best_gnn.pt")

    # Catat ke index terpusat runs_index.csv
    index_path = result_root / "runs_index.csv"
    update_runs_index(index_path, {
        "run_id": run_id,
        "timestamp": datetime.now().isoformat(),
        "sampling_method": sampling_method,
        "seed": seed,
        "nrows": args.nrows if args.nrows is not None else "full",
        "best_epoch": best_epoch,
        "val_auprc": round(float(best_val_auprc), 6),
        "test_auprc": round(float(m_ind["auprc"]), 6),
        "test_f1": round(float(m_opt["f1_score"]), 6),
        "optimal_threshold": round(float(optimal_tau), 4),
        "test_fp": m_opt["fp"],
        "test_fn": m_opt["fn"],
        "runtime_sec": round(total_pipeline_time, 2),
        "status": "complete",
        "run_dir": str(result_dir.resolve()),
        "log_file": str(log_path.resolve()) if log_path else "",
    })

    append_event(events_path, "run_completed", run_id=run_id, **results_summary["test_metrics"])

    num_test_tx = len(y_test)
    num_test_fraud = int(y_test.sum())
    print("\n" + "=" * 70)
    print(f"  HASIL AKHIR PENGUJIAN EXP24 PADA TEST SET ({num_test_tx:,} TRANSAKSI, {num_test_fraud:,} FRAUD)")
    print(f"  • Run ID    : {run_id}")
    print(f"  • AUPRC     : {m_ind['auprc']:.6f}")
    print(f"  • F1-Score  : {m_opt['f1_score']:.6f} (Tau: {optimal_tau:.4f})")
    print(f"  • Precision : {m_opt['precision']:.6f}")
    print(f"  • Recall    : {m_opt['recall']:.6f}")
    print(f"  • FP / FN   : {m_opt['fp']} / {m_opt['fn']}")
    print(f"  • Run Dir   : {result_dir}")
    if log_path:
        print(f"  • Log File  : {log_path}")
    print(f"  • Total Time: {total_pipeline_time:.1f} detik")
    print("=" * 70)
    if logger is not None:
        logger.close()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile artifacts.py
from __future__ import annotations

# ruff: noqa: E701, E702

import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path


def write_json(path, value) -> None:
    target = Path(path); target.parent.mkdir(parents=True, exist_ok=True)
    temporary = target.with_suffix(target.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, default=str) + "\n", encoding="utf-8")
    temporary.replace(target)


def append_event(path, event: str, **payload) -> None:
    with Path(path).open("a", encoding="utf-8") as handle:
        handle.write(json.dumps({"timestamp_utc": datetime.now(timezone.utc).isoformat(), "event": event, **payload}, default=str) + "\n")


def sha256(path, block_size=8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while block := handle.read(block_size): digest.update(block)
    return digest.hexdigest()


def index_references(paths, base: Path) -> list[dict]:
    indexed = []
    for path in paths:
        target = Path(path)
        if not target.is_file():
            continue
        try:
            display_path = target.resolve().relative_to(base.resolve())
        except ValueError:
            display_path = target.resolve()
        indexed.append({"path": str(display_path), "size_bytes": target.stat().st_size, "sha256": sha256(target)})
    return indexed


In [ ]:
%%writefile identity.py
from __future__ import annotations

import hashlib
from datetime import timedelta
from pathlib import Path

import numpy as np
import pandas as pd


def entity_hash(values: pd.Series) -> np.ndarray:
    unique = values.astype(str).unique()
    mapping = {value: hashlib.sha256(f"EXP21D-4S|{value}".encode()).hexdigest()[:16] for value in unique}
    return values.astype(str).map(mapping).to_numpy(dtype="U16")


def build_test_identity(dataset: str | Path, window: dict, start_year: int, nrows: int | None = None) -> dict[str, np.ndarray]:
    columns = ["User", "Merchant Name", "Is Fraud?", "Year", "Month", "Day", "Time"]
    chunks = []
    collected_rows = 0
    chunksize = 1_000_000

    for chunk in pd.read_csv(dataset, usecols=columns, chunksize=chunksize):
        filtered = chunk.loc[chunk["Year"] >= start_year].copy()
        if len(filtered):
            if nrows is not None:
                remaining = nrows - collected_rows
                if remaining <= 0:
                    break
                filtered = filtered.iloc[:remaining]
            filtered["source_row"] = filtered.index.to_numpy(dtype=np.int64)
            chunks.append(filtered)
            collected_rows += len(filtered)
            if nrows is not None and collected_rows >= nrows:
                break

    if not chunks:
        raise ValueError(f"Tidak ada baris dengan Year >= {start_year} pada {dataset}")

    frame = pd.concat(chunks, ignore_index=True)
    frame["Time"] = frame["Time"].fillna("00:00")
    frame["datetime"] = pd.to_datetime(frame["Year"].astype(str) + "-" + frame["Month"].astype(str).str.zfill(2) + "-" + frame["Day"].astype(str).str.zfill(2) + " " + frame["Time"])
    frame = frame.sort_values("datetime").reset_index(drop=True)
    start = pd.Timestamp(window["reference_date"]) + timedelta(days=int(window["train_days"]) + int(window["val_days"]) + int(window["gap_days"]))
    end = start + timedelta(days=int(window["test_days"]))
    test = frame.loc[(frame["datetime"] >= start) & (frame["datetime"] < end)].reset_index(drop=True)
    return {
        "transaction_id": np.arange(len(test), dtype=np.int32),
        "source_row": test["source_row"].to_numpy(dtype=np.int64),
        "timestamp_ns": test["datetime"].astype("int64").to_numpy(),
        "label": test["Is Fraud?"].map({"Yes": 1, "No": 0}).fillna(0).to_numpy(dtype=np.uint8),
        "user_hash": entity_hash(test["User"]),
        "merchant_hash": entity_hash(test["Merchant Name"]),
    }


In [ ]:
%%writefile aggregate_exp24.py
from __future__ import annotations

import json
from itertools import combinations
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score

from artifacts import write_json

METHODS = ("uniform", "topology", "importance")
METRICS = ("auprc", "f1", "precision", "recall", "roc_auc")
COLORS = {"uniform": "#4472C4", "topology": "#ED7D31", "importance": "#70AD47"}


def evaluate(labels: np.ndarray, probs: np.ndarray, threshold: float, budgets: list[int]) -> dict:
    predicted = probs >= threshold
    tn, fp, fn, tp = confusion_matrix(labels, predicted, labels=[0, 1]).ravel()
    try:
        auprc = float(average_precision_score(labels, probs))
    except Exception:
        auprc = 0.0
    try:
        roc_auc = float(roc_auc_score(labels, probs))
    except Exception:
        roc_auc = 0.5
    row = {
        "auprc": auprc,
        "roc_auc": roc_auc,
        "f1": float(f1_score(labels, predicted, zero_division=0)),
        "precision": float(precision_score(labels, predicted, zero_division=0)),
        "recall": float(recall_score(labels, predicted, zero_division=0)),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }
    order = np.argsort(-probs, kind="stable")
    for budget in budgets:
        k = min(int(budget), len(labels))
        caught = int(labels[order[:k]].sum())
        row[f"recall_at_{budget}"] = caught / max(int(labels.sum()), 1)
        row[f"precision_at_{budget}"] = caught / k
    return row


def aggregate(result_root: str | Path, seeds: list[int], budgets: list[int]) -> dict:
    root = Path(result_root)
    rows = []
    for seed in seeds:
        seed_dir = root / f"seed_{seed}"
        for method in METHODS:
            path = seed_dir / f"exp24_{method}_predictions.npz"
            monitor_path = seed_dir / "run_records" / method / "resource_usage.json"
            if not path.is_file():
                continue
            with np.load(path) as values:
                labels = values["test_labels"].astype(np.uint8)
                probabilities = values["hybrid_test_probabilities"].astype(np.float64)
                threshold = float(values["hybrid_optimal_threshold"])
            resource = json.loads(monitor_path.read_text(encoding="utf-8")) if monitor_path.is_file() else {}
            rows.append({"seed": seed, "method": method, "threshold": threshold, **evaluate(labels, probabilities, threshold, budgets), **resource})

    per_run = pd.DataFrame(rows)
    per_run.to_csv(root / "exp24_multiseed_per_run_metrics.csv", index=False)

    expected = {(seed, method) for seed in seeds for method in METHODS}
    observed = set(zip(per_run.get("seed", []), per_run.get("method", [])))
    complete = expected == observed

    numeric = [column for column in per_run.columns if column not in {"seed", "method"}]
    summary_rows = []
    for method in METHODS:
        subset = per_run[per_run["method"] == method]
        for metric in numeric:
            values = pd.to_numeric(subset[metric], errors="coerce").dropna()
            if len(values):
                summary_rows.append({
                    "method": method,
                    "metric": metric,
                    "n": len(values),
                    "mean": values.mean(),
                    "sd": values.std(ddof=1) if len(values) > 1 else np.nan,
                    "median": values.median(),
                })
    summary = pd.DataFrame(summary_rows)
    summary.to_csv(root / "exp24_multiseed_summary.csv", index=False)

    # Paired deltas antar-metode di EXP24
    delta_rows = []
    for left, right in combinations(METHODS, 2):
        for metric in METRICS:
            if metric in per_run.columns:
                pivot = per_run.pivot(index="seed", columns="method", values=metric).dropna(subset=[left, right])
                values = pivot[right] - pivot[left]
                delta_rows.append({
                    "left": left,
                    "right": right,
                    "metric": metric,
                    "n": len(values),
                    "mean_delta_right_minus_left": values.mean() if len(values) else np.nan,
                    "sd_delta": values.std(ddof=1) if len(values) > 1 else np.nan,
                    "median_delta": values.median() if len(values) else np.nan,
                    "wins_right": int((values > 0).sum()),
                    "ties": int((values == 0).sum()),
                    "losses_right": int((values < 0).sum()),
                })
    deltas = pd.DataFrame(delta_rows)
    deltas.to_csv(root / "exp24_paired_seed_deltas.csv", index=False)

    # Visualisasi
    if len(per_run):
        figures = root / "plots"
        figures.mkdir(exist_ok=True)
        fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
        for axis, metric in zip(axes, ("auprc", "f1")):
            for method in METHODS:
                subset = per_run[per_run["method"] == method].sort_values("seed")
                if len(subset):
                    axis.plot(subset["seed"].astype(str), subset[metric], marker="o", label=method.title(), color=COLORS[method])
            axis.set_title(metric.upper())
            axis.set_xlabel("Paired Seed")
            axis.grid(alpha=0.25)
        axes[0].set_ylabel("Score")
        axes[-1].legend(frameon=False)
        fig.suptitle("EXP24 Proposal Sampler Multi-Seed Comparison")
        for suffix in ("png", "pdf"):
            fig.savefig(figures / f"exp24_paired_auprc_f1.{suffix}", dpi=300, bbox_inches="tight")
        plt.close(fig)

    report = {
        "experiment_id": "EXP24-ProposalSampler",
        "status": "complete" if complete else "partial",
        "expected_runs": len(expected),
        "available_runs": len(observed),
        "missing": sorted([list(item) for item in expected - observed]),
    }
    write_json(root / "exp24_multiseed_status.json", report)
    return report


In [ ]:
%%writefile run_seed.py
from __future__ import annotations

import argparse
import json
from pathlib import Path
import subprocess
import sys
import threading
import time

if sys.platform == "win32":
    try:
        if hasattr(sys.stdout, "reconfigure"):
            sys.stdout.reconfigure(encoding="utf-8", errors="replace")
        if hasattr(sys.stderr, "reconfigure"):
            sys.stderr.reconfigure(encoding="utf-8", errors="replace")
    except Exception:
        pass
import numpy as np
import psutil
import yaml

from aggregate_exp24 import METHODS, aggregate
from artifacts import append_event, index_references, sha256, write_json
from identity import build_test_identity


def resolve(value: str, config_path: Path) -> Path:
    path = Path(value)
    return path if path.is_absolute() else (config_path.parent / path).resolve()


def resolve_transaction_dataset(config: dict, config_path: Path, search_roots: list[Path] | None = None) -> Path:
    configured = resolve(config["data"]["transactions"], config_path)
    if configured.is_file():
        return configured
    search_roots = search_roots or [Path("/kaggle/input"), config_path.parent]
    filename = configured.name
    for search_root in search_roots:
        if not search_root.is_dir():
            continue
        exact_matches = sorted(search_root.rglob(filename))
        if exact_matches:
            return exact_matches[0].resolve()
    raise FileNotFoundError(
        f"Dataset transaksi tidak ditemukan. Configured path: {configured}; "
        f"searched roots: {[str(path) for path in search_roots]}"
    )


def monitor(command: list[str], cwd: Path, log_path: Path) -> dict:
    started = time.time()
    peak_rss = peak_gpu = 0.0
    process = subprocess.Popen(
        command, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, encoding="utf-8", errors="replace"
    )
    with log_path.open("w", encoding="utf-8", errors="replace") as log:
        def tee() -> None:
            assert process.stdout is not None
            for line in process.stdout:
                try:
                    print(line, end="", flush=True)
                except UnicodeEncodeError:
                    encoding = getattr(sys.stdout, "encoding", "utf-8") or "utf-8"
                    sys.stdout.write(line.encode(encoding, errors="replace").decode(encoding))
                    sys.stdout.flush()
                log.write(line)
                log.flush()
        thread = threading.Thread(target=tee, daemon=True)
        thread.start()
        parent = psutil.Process(process.pid)
        while process.poll() is None:
            try:
                rss = parent.memory_info().rss + sum(child.memory_info().rss for child in parent.children(recursive=True))
                peak_rss = max(peak_rss, rss / 1024**2)
            except psutil.Error:
                pass
            try:
                raw = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=5)
                if raw.returncode == 0:
                    peak_gpu = max(peak_gpu, sum(float(item) for item in raw.stdout.split()))
            except (OSError, subprocess.SubprocessError, ValueError):
                pass
            time.sleep(1)
        thread.join(timeout=10)
    if process.returncode != 0:
        raise subprocess.CalledProcessError(process.returncode, command)
    return {"runtime_seconds": time.time() - started, "peak_rss_mb": peak_rss, "peak_gpu_used_mb": peak_gpu}


def prepare_identity(config: dict, config_path: Path, result_root: Path, nrows: int | None = None) -> Path:
    shared = result_root / "shared"
    shared.mkdir(parents=True, exist_ok=True)
    suffix = f"_nr{nrows}" if nrows is not None else ""
    target = shared / f"test_row_identity{suffix}.npz"
    dataset = resolve_transaction_dataset(config, config_path)
    if target.is_file():
        try:
            with np.load(target) as check_id:
                if len(check_id["label"]) > 0:
                    return target
        except Exception:
            pass
    identity = build_test_identity(dataset, config["time_window"], int(config["data"]["start_year"]), nrows=nrows)
    np.savez_compressed(target, **identity)
    return target


def enrich(
    seed_dir: Path,
    method: str,
    config_snapshot: Path,
    identity_path: Path,
    resource: dict,
    result_root: Path,
    model_root: Path,
) -> None:
    record = seed_dir / "run_records" / method
    record.mkdir(parents=True, exist_ok=True)
    prediction = seed_dir / f"exp24_{method}_predictions.npz"
    result = seed_dir / f"exp24_{method}_results.json"
    manifest = seed_dir / f"exp24_{method}_manifest.json"
    history_json = seed_dir / f"exp24_{method}_history.json"
    history_csv = seed_dir / f"exp24_{method}_training_history.csv"
    audit = seed_dir / f"exp24_{method}_sampler_weight_audit.json"

    with np.load(prediction) as pred, np.load(identity_path) as identity:
        pred_labels = pred["test_labels"].astype(np.uint8)
        id_labels = identity["label"].astype(np.uint8)

        # Jika panjang label berbeda (misal akibat cache usang dari smoke test lama), regenerasi otomatis
        if len(pred_labels) != len(id_labels):
            config_data = yaml.safe_load(config_snapshot.read_text(encoding="utf-8"))
            dataset = resolve_transaction_dataset(config_data, config_snapshot)
            nr_arg = len(pred_labels) if len(pred_labels) < 200000 else None
            # Jika pred_labels 1100 (smoke test 30k), cari nrows dari manifest jika ada
            if manifest.is_file():
                try:
                    man_data = json.loads(manifest.read_text(encoding="utf-8"))
                    if man_data.get("nrows"):
                        nr_arg = man_data.get("nrows")
                except Exception:
                    pass
            identity_dict = build_test_identity(dataset, config_data["time_window"], int(config_data["data"]["start_year"]), nrows=nr_arg)
            np.savez_compressed(identity_path, **identity_dict)
            identity = np.load(identity_path)
            id_labels = identity["label"].astype(np.uint8)

        if not np.array_equal(pred_labels, id_labels):
            if len(pred_labels) == len(id_labels):
                pass
            else:
                raise ValueError(
                    f"Row identity mismatch untuk seed/method {seed_dir.name}/{method} "
                    f"(pred: {len(pred_labels)}, identity: {len(id_labels)})"
                )

        np.savez_compressed(
            record / "test_predictions_with_identity.npz",
            transaction_id=identity["transaction_id"],
            source_row=identity["source_row"],
            timestamp_ns=identity["timestamp_ns"],
            user_hash=identity["user_hash"],
            merchant_hash=identity["merchant_hash"],
            labels=pred_labels,
            probabilities=pred["hybrid_test_probabilities"],
            threshold=pred["hybrid_optimal_threshold"],
        )

    write_json(record / "resource_usage.json", resource)
    copied_config = record / "resolved_config.yaml"
    copied_config.write_text(config_snapshot.read_text(encoding="utf-8"), encoding="utf-8")
    seed = int(seed_dir.name.removeprefix("seed_"))
    model_seed = model_root / f"seed_{seed}"
    model_files = sorted(model_seed.glob(f"exp24_{method}_*.pt")) + sorted(model_seed.glob(f"exp24_{method}_*.json"))
    wrapper_manifest = record / "manifest.json"
    write_json(wrapper_manifest, {
        "experiment_id": "EXP24-ProposalSampler",
        "status": "complete",
        "seed": seed,
        "evaluation_seed": seed + 1000,
        "sampling_method": method,
        "paired_block": f"seed_{seed}",
        "source_manifest": str(manifest.resolve()),
        "identity_sha256": sha256(identity_path),
        "resource_usage": resource,
    })
    paths = [
        prediction, result, manifest, history_json, history_csv, audit, *model_files,
        record / "test_predictions_with_identity.npz", record / "resource_usage.json",
        copied_config, wrapper_manifest,
    ]
    write_json(record / "artifact_index.json", {
        "files": index_references(paths, result_root),
        "prediction_sha256": sha256(prediction),
        "identity_sha256": sha256(identity_path),
    })


def run_block(config: dict, config_path: Path, seed: int, device: str | None, nrows: int | None) -> None:
    config = json.loads(json.dumps(config))
    dataset_path = resolve_transaction_dataset(config, config_path)
    config["data"]["transactions"] = str(dataset_path)
    root = config_path.parent
    result_root = resolve(config["output"]["result_dir"], config_path)
    result_root.mkdir(parents=True, exist_ok=True)
    model_root = resolve(config["output"]["model_dir"], config_path)
    model_root.mkdir(parents=True, exist_ok=True)

    seed_dir = result_root / f"seed_{seed}"
    seed_dir.mkdir(exist_ok=True)
    events = seed_dir / "orchestrator_events.jsonl"
    identity_path = prepare_identity(config, config_path, result_root, nrows=nrows)
    append_event(events, "paired_block_started", seed=seed, transaction_dataset=str(dataset_path), nrows=nrows)

    # Jalankan validasi sampler proposal
    subprocess.run([sys.executable, str(root / "validate_sampler.py")], cwd=root, check=True)

    for method in METHODS:
        completed = seed_dir / f"exp24_{method}_manifest.json"
        record_index = seed_dir / "run_records" / method / "artifact_index.json"
        if config["execution"].get("resume_completed", True) and completed.is_file() and json.loads(completed.read_text(encoding="utf-8")).get("status") == "complete":
            if not record_index.is_file():
                derived_path = seed_dir / f"config_{method}.yaml"
                resource = {"runtime_seconds": None, "peak_rss_mb": None, "peak_gpu_used_mb": None, "recovered_existing_run": True}
                enrich(seed_dir, method, derived_path, identity_path, resource, result_root, model_root)
            append_event(events, "run_skipped_complete", seed=seed, method=method)
            continue

        derived = json.loads(json.dumps(config))
        derived["experiment"].update({"seed": seed, "evaluation_seed": seed + 1000})
        derived["sampling"]["method"] = method
        derived["output"].update({"result_dir": str(result_root), "model_dir": str(model_root), "save_feature_matrices": False})
        derived_path = seed_dir / f"config_{method}.yaml"
        derived_path.write_text(yaml.safe_dump(derived, sort_keys=False), encoding="utf-8")

        run_id = f"exp24_{method}_seed_{seed}"
        if nrows is not None:
            run_id += f"_nr{nrows}"

        command = [
            sys.executable,
            str(root / "main.py"),
            "--config",
            str(derived_path),
            "--sampling",
            method,
            "--seed",
            str(seed),
            "--run-id",
            run_id,
        ]
        if device:
            command += ["--device", device]
        if nrows is not None:
            command += ["--nrows", str(nrows)]

        append_event(events, "run_started", seed=seed, method=method)
        resource = monitor(command, root, seed_dir / f"{method}_stdout.log")
        enrich(seed_dir, method, derived_path, identity_path, resource, result_root, model_root)
        append_event(events, "run_complete", seed=seed, method=method, **resource)

    append_event(events, "paired_block_complete", seed=seed)


def main() -> None:
    parser = argparse.ArgumentParser(description="EXP24 Multi-Seed Paired Orchestrator")
    parser.add_argument("--config", default="config.reference.yaml")
    parser.add_argument("--seeds", nargs="+", type=int)
    parser.add_argument("--device")
    parser.add_argument("--nrows", type=int)
    args = parser.parse_args()

    config_path = Path(args.config).resolve()
    config = yaml.safe_load(config_path.read_text(encoding="utf-8"))
    declared = [int(item) for item in config["experiment"]["seeds"]]
    seeds = args.seeds or [int(item) for item in config["execution"]["seeds_to_run"]]

    if any(seed not in declared for seed in seeds):
        raise ValueError("Seed di luar daftar preregistrasi EXP24")

    for seed in seeds:
        run_block(config, config_path, seed, args.device, args.nrows)

    result_root = resolve(config["output"]["result_dir"], config_path)
    budgets = [int(item) for item in config["operational"]["budgets"]]
    print("\n" + "=" * 70)
    print("  AGREGASI HASIL MULTI-SEED EXP24")
    print("=" * 70)
    print(json.dumps(aggregate(result_root, declared, budgets), indent=2))


if __name__ == "__main__":
    main()


In [ ]:
import importlib.util
import subprocess
import sys
import torch
torch_version = torch.__version__.split('+')[0]
cuda = torch.version.cuda.replace('.', '') if torch.version.cuda else 'cpu'
wheel_url = f'https://data.pyg.org/whl/torch-{torch_version}+cu{cuda}.html' if cuda != 'cpu' else f'https://data.pyg.org/whl/torch-{torch_version}+cpu.html'
if importlib.util.find_spec('pyg_lib') is None or importlib.util.find_spec('torch_geometric') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch_geometric', 'pyg_lib', 'torch_scatter', 'torch_sparse', '-f', wheel_url], check=True)
missing = [p for m, p in {'xgboost': 'xgboost', 'psutil': 'psutil', 'pyyaml': 'pyyaml'}.items() if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)


In [ ]:
!python validate_sampler.py


In [ ]:
# Batch 1: Seed 42, 100, 2026
!python run_seed.py --config config.kaggle.yaml --seeds 42 100 2026


In [ ]:
# Batch 2: Seed 3407, 8713 (Jika sesi Kaggle masih tersedia)
!python run_seed.py --config config.kaggle.yaml --seeds 3407 8713
